<div style="background-color:#166534; color:#fde047; padding: 25px; border-radius: 10px; text-align:left;">
  <h1 style="font-size:26px; font-family:calibri;"><b>📦 From Clicks to Carts: Forecasting Amazon Electronics Demand (2025)</b></h1>
  <p style="font-size:18px; font-family:calibri; line-height:1.6em;">
    Turning marketplace scraps into a clean, predictive pipeline for monthly demand — price-smart, title-aware, and ready for business impact.
  </p>
</div>

<div style="background-color:#166534; color:#fde047; padding: 20px; border-radius: 10px;">
  <h2 style="font-size:22px; font-family:calibri;"><b>🧠 Introduction</b></h2>
  <p style="font-size:18px; font-family:calibri; line-height:1.6em;">
    Shoppers don’t read spreadsheets—they scan titles, skim ratings, and pounce on deals. Brands wonder: Which products will surge this month?
  </p>
  <p style="font-size:18px; font-family:calibri;">
    In this notebook, we turn noisy marketplace data into a monthly demand signal. From titles and prices to reviews and ratings, we extract meaningful signals to forecast product traction — fast, reproducible, and e-commerce ready.
  </p>
</div>

<div style="background-color:#166534; color:#fde047; padding: 20px; border-radius: 10px;">
  <h2 style="font-size:22px; font-family:calibri;"><b>📌 Problem Statement</b></h2>
  <p style="font-size:18px; font-family:calibri; line-height:1.6em;">
    Predict monthly product demand (proxy: "bought in the past month") using scraped Amazon Electronics data.
  </p>
  <ul style="font-size:18px; font-family:calibri; line-height:1.8em;">
    <li>Challenges: duplicated URLs, string-typed numerics, missing prices</li>
    <li>Heavy text fields (titles, URLs), time-variance, inconsistent formatting</li>
    <li>Target: log1p(bought_last_month), modeled as regression</li>
  </ul>
</div>

<div style="background-color:#166534; color:#fde047; padding: 20px; border-radius: 10px;">
  <h2 style="font-size:22px; font-family:calibri;"><b>📁 About the Dataset</b></h2>
  <ul style="font-size:18px; font-family:calibri; line-height:1.8em;">
    <li><b>Source:</b> Amazon Electronics Products Sales (2025)</li>
    <li><b>Size:</b> ~42K rows (raw) → 9.5K rows modeled (cleaned)</li>
    <li><b>Features:</b> title, rating, reviews, prices (x3), ASIN, URL, collected_at</li>
    <li><b>Challenges:</b> all strings, duplicate URLs, ad-tracking, heavy missingness</li>
  </ul>
</div>

<div style="background-color:#166534; color:#fde047; padding: 20px; border-radius: 10px;">
  <h2 style="font-size:22px; font-family:calibri;"><b>🛠️ Project Workflow</b></h2>
  <ul style="font-size:18px; font-family:calibri; line-height:1.8em;">
    <li>🧼 <b>Parse Targets:</b> Converted “20K+ bought” → integer + log1p</li>
    <li>🔧 <b>Fix & Impute Prices:</b> fallback: discounted → variant → original</li>
    <li>🧹 <b>Deduplicate:</b> ASIN-level deduplication (latest snapshot kept)</li>
    <li>📉 <b>Feature Engineering:</b> discount_pct, final_price, log1p_reviews</li>
    <li>⏳ <b>Time-Based CV:</b> 5 bins → 4 expanding validation folds</li>
    <li>📊 <b>Models:</b> LightGBM + TF-IDF → CatBoost (native text) ✅</li>
    <li>📈 <b>Final Model:</b> CatBoost (full training), RMSE(log) ≈ 0.996</li>
    <li>🖼 <b>EDA:</b> time trends, rating/price effects, discount buckets</li>
  </ul>
</div>

<div style="background-color:#166534; color:#fde047; padding: 20px; border-radius: 10px;">
  <h2 style="font-size:22px; font-family:calibri;"><b>👨‍💻 About the Author</b></h2>
  <p style="font-size:18px; font-family:calibri; line-height:1.6em;">
    Hi! I'm <b>Denver Magtibay</b>, an Agentic AI Engineer and Electronics Engineer based in the Philippines. I build ML systems that connect signals to action — from flood alerts to e-commerce forecasting.
  </p>
  <p style="font-size:18px; font-family:calibri;">
    My Kaggle notebooks are clean, actionable, and designed to teach through building.
  </p>
  <p style="font-size:18px; font-family:calibri;">
    🔗 <b>Connect:</b>
    <a href="https://www.linkedin.com/in/denvermagtibay/" target="_blank" style="color:#fef9c3;">LinkedIn</a> |
    <a href="https://www.kaggle.com/denvermagtibay" target="_blank" style="color:#fef9c3;">Kaggle</a> |
    <a href="mailto:engr.denver.magtibay@gmail.com" style="color:#fef9c3;">Email</a>
  </p>
</div>

# STEP 1 — Data audit

In [ ]:
import warnings, gc
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

PATH = "/kaggle/input/amazon-products-sales-dataset-42k-items-2025/amazon_products_sales_data_uncleaned.csv"

def mem_usage(df: pd.DataFrame) -> str:
    return f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"

def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    m = df.isna().sum().sort_values(ascending=False)
    out = pd.DataFrame({"missing": m, "missing_pct": (m / len(df) * 100).round(2)})
    return out[out.missing > 0].head(30)

def sample_values(df: pd.DataFrame, cols, n=20):
    for c in cols:
        if c in df.columns:
            vals = (df[c].dropna().astype(str).unique().tolist())[:n]
            print(f"\n▶ Sample values — {c} ({len(vals)} shown):")
            print(", ".join(vals) if vals else "[no non-null examples]")

print(f"Using file: {PATH}")
try:
    df = pd.read_csv(PATH, engine="pyarrow", low_memory=False)
except Exception:
    df = pd.read_csv(PATH, low_memory=False)

print(f"Shape: {df.shape} | Memory: {mem_usage(df)}")
print("\nColumns:", list(df.columns))

In [ ]:
# Dtypes
print("\nDtypes:")
print(df.dtypes)

In [ ]:
# Target check (handles your naming)
ALT_TARGETS = ["purchased_last_month", "bought_in_last_month", "bought_last_month"]
present_targets = [t for t in ALT_TARGETS if t in df.columns]
if present_targets:
    tgt = present_targets[0]
    non_null = df[tgt].notna().sum()
    unique_vals = df[tgt].nunique(dropna=True)
    print(f"\nTarget candidate '{tgt}': non-null={non_null}, unique={unique_vals}")
    print("Target preview (10):", df[tgt].dropna().head(10).tolist())
else:
    print("\n[WARN] No target-like column from", ALT_TARGETS)

In [ ]:
# Missingness (top 30)
print("\nTop missing columns:")
display(missing_report(df))

# Duplicate signals
for key in ["product_url", "title"]:
    if key in df.columns:
        dup_cnt = df.duplicated(subset=[key]).sum()
        print(f"\nPotential duplicates by {key}: {dup_cnt}")

In [ ]:
# Title length stats
if "title" in df.columns:
    tl = df["title"].astype(str).str.len()
    print("\nTitle length (chars):",
          f"mean={tl.mean():.1f}, std={tl.std():.1f}, p50={tl.quantile(0.50):.1f}, p95={tl.quantile(0.95):.1f}, max={tl.max():.0f}")

# Insights

All 16 cols are strings; we’ll need coercion.

Target candidate bought_in_last_month is text like “6K+ bought in past month” (≈59 unique tokens).

Heavy dup signals: title has 33,867 dups; product_url has 2,068 dups and 4.85% missing.

Big gaps: sustainability_badges (92% missing), buy_box_availability (34%), prices & delivery ≈27% missing.

# Parse target only (safe & focused)
# Creates: purchased_last_month (float), target_log1p (float), collected_at_dt (datetime)

In [ ]:
import re, unicodedata, numpy as np, pandas as pd
from IPython.display import display

# Ensure df is loaded
if 'df' not in globals():
    PATH = "/kaggle/input/amazon-products-sales-dataset-42k-items-2025/amazon_products_sales_data_uncleaned.csv"
    df = pd.read_csv(PATH, low_memory=False)

def _safe_float(s: str):
    try:
        return float(s)
    except Exception:
        return np.nan

def _num_with_suffix_to_float(num_str: str, suf: str) -> float:
    if num_str is None:
        return np.nan
    num_str = str(num_str).replace(',', '').strip()
    if num_str == '':
        return np.nan
    num = _safe_float(num_str)
    if np.isnan(num):
        return np.nan
    suf = (suf or '').lower()
    if suf == 'k':
        num *= 1_000
    elif suf == 'm':
        num *= 1_000_000
    return num

def _clean_text(x):
    if pd.isna(x):
        return ""
    # Normalize unicode and collapse spaces
    s = unicodedata.normalize("NFKC", str(x)).strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return s

def parse_bought_in_last_month(s: object) -> float:
    """
    Handles:
      '6K+ bought in past month', '300+ bought in past month',
      '1.5k+ ...', '100k+ ...', 'Less than 100 ...',
      'New to market'/'Just launched' -> 0,
      blank/garbage -> NaN.
    """
    t = _clean_text(s)
    if t == "":
        return np.nan

    # Zero-ish phrases
    if any(z in t for z in ('new to market', 'just launched', 'be the first')):
        return 0.0

    # "Less than X [k/m]"
    m = re.search(r'less than\s*([\d,]+(?:\.\d+)?)\s*([km]?)', t)
    if m:
        base = _num_with_suffix_to_float(m.group(1), m.group(2))
        return 0.5 * base if base == base else np.nan  # midpoint heuristic

    # General "{X}[k/m]? (+)?"
    m = re.search(r'([\d,]+(?:\.\d+)?)\s*([km]?)\s*\+?', t)
    if m:
        return _num_with_suffix_to_float(m.group(1), m.group(2))

    # Fallback: first number seen
    m = re.search(r'(\d[\d,]*(?:\.\d+)?)', t)
    if m:
        return _safe_float(m.group(1).replace(',', ''))

    return np.nan

# --- Apply robust parsing ---
parsed = df['bought_in_last_month'].apply(parse_bought_in_last_month)
df['purchased_last_month'] = pd.to_numeric(parsed, errors='coerce')
df['target_log1p'] = np.log1p(df['purchased_last_month'])

# Parse collected_at to datetime (safe)
df['collected_at_dt'] = pd.to_datetime(df.get('collected_at', pd.Series(index=df.index)),
                                       errors='coerce', infer_datetime_format=True)

# --- QC report ---
n = len(df)
raw_non_null = df['bought_in_last_month'].notna().sum()
empty_strs = (df['bought_in_last_month'].astype(str).str.strip() == '').sum()
parsed_non_null = df['purchased_last_month'].notna().sum()
zero_ct = int((df['purchased_last_month'] == 0).sum())

print("=== Target Parse Report ===")
print(f"Rows: {n}")
print(f"Original non-null strings: {raw_non_null}")
print(f"Blank/empty strings: {empty_strs}")
print(f"Parsed to numeric: {parsed_non_null}")
print(f"Zeros after parsing (incl. 'new/just launched'): {zero_ct}")

if parsed_non_null > 0:
    desc = df.loc[df['purchased_last_month'].notna(), 'purchased_last_month'].describe(percentiles=[.01,.05,.5,.9,.95,.99])
    print("\nNumeric target distribution (purchased_last_month):")
    display(desc.to_frame(name='value'))

    log_desc = df.loc[df['target_log1p'].notna(), 'target_log1p'].describe(percentiles=[.01,.05,.5,.9,.95,.99])
    print("Log1p(target) distribution:")
    display(log_desc.to_frame(name='value'))

print("\nTop 12 raw strings in 'bought_in_last_month':")
display(df['bought_in_last_month'].value_counts(dropna=True).head(12))

bad = df[df['bought_in_last_month'].notna() & df['purchased_last_month'].isna()]
if len(bad):
    print("Sample of unparsed patterns (up to 10):")
    display(bad['bought_in_last_month'].drop_duplicates().head(10).to_frame())

dt_non_null = df['collected_at_dt'].notna().sum()
if dt_non_null:
    print(f"\nCollected_at parsed: {dt_non_null}/{n} rows")
    print(f"Date range: {df['collected_at_dt'].min()} -> {df['collected_at_dt'].max()}")
else:
    print("\n[Warn] No parsable dates in 'collected_at'.")

print("\n✅ Done (robust): created/updated ['purchased_last_month','target_log1p','collected_at_dt'].")

# Quick Takeaways

Parsed to numeric: 32,233 / 39,458 non-null strings → 7,225 were non-numeric phrases (e.g., “No featured offers available”, “Typical:”, “List”, “More Buying Choices”).

Heavy right tail (max 100K), median ≈ 200, log1p median ≈ 5.30.

Dates look good for time-based CV: 2025-08-21 → 2025-08-30.

# Type coercion + renaming (safe)
# Creates: dfc (standardized & typed)

In [ ]:
import re, unicodedata, numpy as np, pandas as pd
from IPython.display import display

# Ensure df is present (from previous step). If not, load fresh.
if 'df' not in globals():
    PATH = "/kaggle/input/amazon-products-sales-dataset-42k-items-2025/amazon_products_sales_data_uncleaned.csv"
    df = pd.read_csv(PATH, low_memory=False)

# ---------- helpers ----------
def _clean_str(x):
    if pd.isna(x): return np.nan
    s = unicodedata.normalize("NFKC", str(x)).strip()
    return re.sub(r'\s+', ' ', s)

def first_number(text):
    if pd.isna(text): return np.nan
    m = re.search(r'(\d+[\d,]*\.?\d*)', str(text))
    if not m: return np.nan
    try: return float(m.group(1).replace(',', ''))
    except: return np.nan

def parse_knum(x):
    """e.g. '1,234', '1.5k', '12K+', '3m' -> float"""
    if pd.isna(x): return np.nan
    s = str(x).strip().lower().replace(',', '')
    m = re.match(r'(\d+(\.\d+)?)([km])?', s)
    if not m: return np.nan
    num = float(m.group(1))
    suf = m.group(3)
    if suf == 'k': num *= 1_000
    elif suf == 'm': num *= 1_000_000
    return num

def parse_price(x):
    """
    Extract first numeric token from price-like strings.
    Handles currencies, commas, and ranges like '₹1,399 – ₹1,599'.
    """
    if pd.isna(x): return np.nan
    s = str(x)
    nums = re.findall(r'\d[\d,]*(?:\.\d+)?', s)
    if not nums: return np.nan
    try: return float(nums[0].replace(',', ''))
    except: return np.nan

def parse_boolish(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    truthy = {'true','t','1','yes','y','available','best seller','bestseller','sponsored','coupon','has coupon','in stock'}
    falsy  = {'false','f','0','no','n','unavailable','not available','out of stock','none'}
    if s in truthy: return True
    if s in falsy:  return False
    # try numeric cast
    try: return bool(int(float(s)))
    except: return np.nan

# ---------- rename ----------
rename_map = {
    'title':'product_title',
    'rating':'product_rating',
    'number_of_reviews':'total_reviews',
    'bought_in_last_month':'bought_in_last_month',  # keep original text (target already parsed previously)
    'current/discounted_price':'discounted_price',
    'listed_price':'original_price',
    'is_couponed':'has_coupon',
    'delivery_details':'delivery_date',
    'sustainability_badges':'sustainability_tags',
    'image_url':'product_image_url',
    'product_url':'product_page_url',
    'collected_at':'data_collected_at',
}
dfc = df.rename(columns=rename_map).copy()

# ---------- coercions ----------
# text cleanup
if 'product_title' in dfc:
    dfc['product_title'] = dfc['product_title'].map(_clean_str)

# rating (clip to [0,5])
if 'product_rating' in dfc:
    dfc['product_rating'] = dfc['product_rating'].map(first_number).clip(0, 5)

# reviews
if 'total_reviews' in dfc:
    dfc['total_reviews'] = dfc['total_reviews'].map(parse_knum).astype('float')

# prices
for c in ['discounted_price','original_price','price_on_variant']:
    if c in dfc:
        dfc[c] = dfc[c].map(parse_price).astype('float')

# booleans
for c in ['is_best_seller','is_sponsored','has_coupon','buy_box_availability']:
    if c in dfc:
        dfc[c] = dfc[c].map(parse_boolish)

# dataset note: NaN in buy_box_availability -> False (no button)
if 'buy_box_availability' in dfc:
    dfc['buy_box_availability'] = dfc['buy_box_availability'].fillna(False).astype(bool)

# dates
for c_old, c_new in [('data_collected_at','data_collected_at'), ('delivery_date','delivery_date')]:
    if c_old in dfc:
        dfc[c_new] = pd.to_datetime(dfc[c_old], errors='coerce', infer_datetime_format=True)

# keep numeric target from previous step if present; otherwise leave for later
if 'purchased_last_month' in dfc.columns:
    pass
elif 'purchased_last_month' in df.columns:
    dfc['purchased_last_month'] = df['purchased_last_month']
if 'target_log1p' in df.columns and 'target_log1p' not in dfc.columns:
    dfc['target_log1p'] = df['target_log1p']
if 'collected_at_dt' in df.columns and 'data_collected_at' in dfc.columns:
    # prefer already parsed; else ensure we have a parsed datetime field
    dfc['collected_at_dt'] = df['collected_at_dt']
elif 'data_collected_at' in dfc.columns:
    dfc['collected_at_dt'] = pd.to_datetime(dfc['data_collected_at'], errors='coerce', infer_datetime_format=True)

# ---------- derived fields ----------
if {'discounted_price','original_price'} <= set(dfc.columns):
    mask = (dfc['original_price'] > 0)
    dfc['discount_percentage'] = np.where(mask,
        (dfc['original_price'] - dfc['discounted_price']) / dfc['original_price'] * 100,
        np.nan
    )
    dfc['has_discount'] = (dfc['discounted_price'].notna()) & (dfc['original_price'].notna()) & (dfc['discounted_price'] < dfc['original_price'])
    dfc['price_anomaly_discount_gt_original'] = (dfc['discounted_price'].notna()) & (dfc['original_price'].notna()) & (dfc['discounted_price'] > dfc['original_price'])

# ---------- QC summary ----------
print("After coercion & renaming")
print("Shape:", dfc.shape)

print("\nDtypes:")
print(dfc.dtypes)

num_cols = [c for c in ['product_rating','total_reviews','discounted_price','original_price','price_on_variant',
                        'discount_percentage','purchased_last_month','target_log1p'] if c in dfc.columns]
if num_cols:
    print("\nNumeric summary:")
    display(dfc[num_cols].describe(percentiles=[.01,.05,.5,.9,.95,.99]).T)

print("\nBoolean counts:")
for c in ['is_best_seller','is_sponsored','has_coupon','buy_box_availability','has_discount','price_anomaly_discount_gt_original']:
    if c in dfc.columns:
        print(f"{c}:\n{dfc[c].value_counts(dropna=False)}\n")

if {'discounted_price','original_price'} <= set(dfc.columns):
    weird = int((dfc['discounted_price'] > dfc['original_price']).sum())
    both_nan = int(dfc['discounted_price'].isna().sum() + dfc['original_price'].isna().sum())
    print(f"Rows where discounted_price > original_price: {weird}")
    print(f"Missing counts — discounted_price: {dfc['discounted_price'].isna().sum()}, original_price: {dfc['original_price'].isna().sum()}")

print("\nTop missing columns (head):")
m = dfc.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({'missing': m, 'missing_pct': (m/len(dfc)*100).round(2)}).head(12))

print("\nPeek:")
cols_show = [c for c in ['product_title','product_rating','total_reviews','purchased_last_month',
                         'discounted_price','original_price','discount_percentage',
                         'is_best_seller','is_sponsored','has_coupon','buy_box_availability',
                         'delivery_date','product_image_url','product_page_url','data_collected_at'] if c in dfc.columns]
display(dfc[cols_show].head(3))

print("\n✅ dfc ready (typed + renamed). No rows dropped, no dedup yet.")

# De-dup by product_page_url (keep latest)
# Creates: dfd (deduplicated)

In [ ]:
import re, numpy as np, pandas as pd
from urllib.parse import urlsplit, parse_qs, unquote
from IPython.display import display

# Start from standardized frame
base = 'dfc' if 'dfc' in globals() else 'df'
df0 = globals()[base].copy()

ASIN_RE_PATH = re.compile(r"/(?:dp|gp/product|product)/([A-Z0-9]{10})(?:[/?]|$)", re.I)
ASIN_RE_ANY  = re.compile(r"(^|[^A-Z0-9])([A-Z0-9]{10})(?=$|[^A-Z0-9])")  # last-resort, guarded

def extract_asin(u: object) -> str | None:
    """Extract ASIN from Amazon product URLs:
       1) Path patterns: /dp/ASIN, /gp/product/ASIN, /product/ASIN
       2) Query params: asin=... or a nested 'url=' pointing to a dp link
       Conservative: if we can't confidently find a 10-char ASIN => None.
    """
    if pd.isna(u): return None
    s = str(u).strip()
    if not s: return None
    try:
        parts = urlsplit(s)
        path  = parts.path or "/"
        query = parts.query or ""

        # 1) Path pattern
        m = ASIN_RE_PATH.search(path)
        if m:
            return m.group(1).upper()

        # 2a) Direct query key
        q = parse_qs(query)
        for key in ('asin', 'ASIN'):
            if key in q and len(q[key]):
                cand = q[key][0].strip()
                if re.fullmatch(r"[A-Z0-9]{10}", cand, re.I):
                    return cand.upper()

        # 2b) Nested url=... that itself contains /dp/ASIN
        if 'url' in q and len(q['url']):
            nested = unquote(q['url'][0])
            m2 = ASIN_RE_PATH.search(nested)
            if m2:
                return m2.group(1).upper()

        # Final conservative fallback: search entire URL for 10-char, only if host looks like amazon
        host = (parts.netloc or "").lower()
        if "amazon." in host:
            m3 = ASIN_RE_ANY.search(s)
            if m3:
                cand = m3.group(2).upper()
                # guard: avoid obvious non-ASINs (all digits etc. is fine; ASIN can be alnum)
                if re.fullmatch(r"[A-Z0-9]{10}", cand):
                    return cand

    except Exception:
        return None
    return None

# Build ASIN key
df0['asin_key'] = df0.get('product_page_url', pd.Series(index=df0.index, dtype=object)).map(extract_asin)

# Partition
with_asin = df0['asin_key'].notna()
df_with = df0[with_asin].copy()      # candidates to dedup
df_none = df0[~with_asin].copy()     # keep as-is

# Helper sort columns
if 'collected_at_dt' not in df_with.columns and 'data_collected_at' in df_with.columns:
    df_with['collected_at_dt'] = pd.to_datetime(df_with['data_collected_at'], errors='coerce', infer_datetime_format=True)

df_with['_has_dt']  = df_with['collected_at_dt'].notna().astype(int)
df_with['_sort_dt'] = df_with['collected_at_dt'].fillna(pd.Timestamp('1970-01-01'))

info_cols = [c for c in ['product_rating','total_reviews','discounted_price','original_price',
                         'price_on_variant','purchased_last_month','product_title'] if c in df_with.columns]
df_with['_info_score'] = df_with[info_cols].notna().sum(axis=1).astype(int)

df_with['_has_disc'] = (
    df_with.get('discounted_price').notna()
    & df_with.get('original_price').notna()
    & (df_with.get('discounted_price') < df_with.get('original_price'))
).fillna(False).astype(int)

df_with['_reviews_sort'] = df_with.get('total_reviews', pd.Series(index=df_with.index, dtype='float64')).fillna(-1e18)
df_with['_target_sort']  = df_with.get('purchased_last_month', pd.Series(index=df_with.index, dtype='float64')).fillna(-1e18)

# Sort + choose canonical per ASIN
order_cols = ['asin_key','_has_dt','_sort_dt','_info_score','_has_disc','_reviews_sort','_target_sort']
df_with_sorted = df_with.sort_values(order_cols, ascending=[True, False, False, False, False, False, False])
df_keep = df_with_sorted.drop_duplicates(subset=['asin_key'], keep='first').copy()

# Combine back
dfd = pd.concat([df_keep, df_none], axis=0, ignore_index=True)

# Reports
n_total     = len(df0)
n_with_asin = len(df_with)
n_unique    = df_with['asin_key'].nunique()
n_groups    = df_with.groupby('asin_key').size()
n_dup_groups= int((n_groups > 1).sum())
n_no_asin   = len(df_none)
n_kept      = len(df_keep)
n_dropped   = n_with_asin - n_kept
n_after     = len(dfd)

print("=== Safe De-dup by ASIN only ===")
print(f"Rows before: {n_total}")
print(f"Rows with ASIN: {n_with_asin} | unique ASINs: {n_unique} | duplicate ASIN groups: {n_dup_groups}")
print(f"Rows without ASIN (kept as-is): {n_no_asin}")
print(f"Rows kept from ASIN groups: {n_kept}")
print(f"Duplicates removed (ASIN-only): {n_dropped}")
print(f"Rows after: {n_after}")

# Sanity checks
try:
    assert n_after == n_no_asin + n_unique
    assert dfd['asin_key'].dropna().is_unique
except AssertionError:
    print("\n[Warn] Sanity check failed; counts may not align; proceed but review.")

# Top duplicate ASIN groups
dup_sizes = n_groups[n_groups > 1].sort_values(ascending=False).head(10).to_frame('size')
print("\nTop duplicate ASIN groups (size > 1):")
display(dup_sizes)

# Display one sample ASIN group
if len(dup_sizes):
    sample_asin = dup_sizes.index[0]
    grp = df_with[df_with['asin_key'] == sample_asin].copy()
    grp_sorted = grp.sort_values(['_has_dt','_sort_dt','_info_score','_has_disc','_reviews_sort','_target_sort'],
                                 ascending=[False, False, False, False, False, False])
    cols_show = [c for c in ['asin_key','product_title','collected_at_dt','discounted_price','original_price',
                             'total_reviews','purchased_last_month'] if c in grp_sorted.columns]
    print(f"\nSample ASIN group: {sample_asin}")
    display(grp_sorted[cols_show].head(8))

# Clean helper columns from final frame
helper_cols = ['_has_dt','_sort_dt','_info_score','_has_disc','_reviews_sort','_target_sort']
dfd.drop(columns=[c for c in helper_cols if c in dfd.columns], inplace=True)

# Save
dfd.to_parquet("amazon_dedup_asin.parquet", index=False)
print("\n✅ dfd ready: ASIN-only dedup applied. Saved -> amazon_dedup_asin.parquet")

# Target hygiene & modeling frame
# Creates: dfm  (clean subset for modeling)

In [ ]:
import re, unicodedata, numpy as np, pandas as pd
from IPython.display import display

# ---- Config ----
CAP_PCT   = 0.995   # winsorize at the 99.5th percentile
MIN_UNITS = 1e-9    # treat <= this as missing

# pick deduped frame if present
base = 'dfd' if 'dfd' in globals() else ('dfc' if 'dfc' in globals() else 'df')
df1 = globals()[base].copy()

# ensure a parsed target exists; if not, reconstruct from text
def _safe_float(s):
    try: return float(s)
    except: return np.nan

def _num_with_suffix_to_float(num_str: str, suf: str) -> float:
    if num_str is None: return np.nan
    x = str(num_str).replace(',', '').strip()
    if x == '': return np.nan
    num = _safe_float(x)
    if not np.isnan(num):
        suf = (suf or '').lower()
        if suf == 'k': num *= 1_000
        elif suf == 'm': num *= 1_000_000
    return num

def parse_bought_in_last_month(s: object) -> float:
    if pd.isna(s): return np.nan
    t = unicodedata.normalize("NFKC", str(s)).strip().lower()
    if t == "": return np.nan
    if any(z in t for z in ('new to market', 'just launched', 'be the first')):
        return 0.0
    m = re.search(r'less than\s*([\d,]+(?:\.\d+)?)\s*([km]?)', t)
    if m:
        base = _num_with_suffix_to_float(m.group(1), m.group(2))
        return 0.5*base if base==base else np.nan
    m = re.search(r'([\d,]+(?:\.\d+)?)\s*([km]?)\s*\+?', t)
    if m:
        return _num_with_suffix_to_float(m.group(1), m.group(2))
    m = re.search(r'(\d[\d,]*(?:\.\d+)?)', t)
    if m:
        return _safe_float(m.group(1).replace(',', ''))
    return np.nan

if 'purchased_last_month' not in df1.columns and 'bought_in_last_month' in df1.columns:
    df1['purchased_last_month'] = pd.to_numeric(df1['bought_in_last_month'].map(parse_bought_in_last_month), errors='coerce')

# basic date safety
if 'collected_at_dt' not in df1.columns and 'data_collected_at' in df1.columns:
    df1['collected_at_dt'] = pd.to_datetime(df1['data_collected_at'], errors='coerce', infer_datetime_format=True)

# ---- Filter to rows with a usable target ----
n_before = len(df1)
mask_num = df1['purchased_last_month'].notna() & (df1['purchased_last_month'] > MIN_UNITS)
dfm = df1.loc[mask_num].copy()
n_after_num = len(dfm)

# ---- Outlier capping & target transform ----
q_hi = float(dfm['purchased_last_month'].quantile(CAP_PCT)) if n_after_num else np.nan
dfm['purchased_last_month_capped'] = np.minimum(dfm['purchased_last_month'], q_hi) if n_after_num else np.nan
dfm['y'] = np.log1p(dfm['purchased_last_month_capped'])

# helper for stratified CV (5 quantile bins over y)
try:
    dfm['cv_strat5'] = pd.qcut(dfm['y'], q=5, labels=False, duplicates='drop')
except Exception:
    dfm['cv_strat5'] = 0

# ---- Report ----
print("=== Target Hygiene & Modeling Frame ===")
print(f"Rows in input frame ({base}): {n_before}")
print(f"Rows with numeric target > 0: {n_after_num}  ({n_after_num/n_before*100:.1f}%)")
if n_after_num:
    print(f"99.5th pct cap value: {q_hi:,.0f}")
    desc_pre  = df1['purchased_last_month'].dropna().describe(percentiles=[.5,.9,.95,.99])
    desc_post = dfm['purchased_last_month_capped'].describe(percentiles=[.5,.9,.95,.99])
    print("\nTarget (pre-filter) summary:")
    display(desc_pre.to_frame('value'))
    print("Target (post-cap) summary:")
    display(desc_post.to_frame('value'))
    print("y = log1p(target_capped) summary:")
    display(dfm['y'].describe(percentiles=[.5,.9,.95,.99]).to_frame('value'))

# quick peek
cols_show = [c for c in ['product_title','purchased_last_month','purchased_last_month_capped','y',
                         'product_rating','total_reviews',
                         'discounted_price','original_price','discount_percentage',
                         'is_best_seller','is_sponsored','buy_box_availability',
                         'collected_at_dt','product_page_url'] if c in dfm.columns]
print("\nPeek at dfm:")
display(dfm[cols_show].head(5))

# save
dfm.to_parquet("amazon_modeling_frame.parquet", index=False)
print("\n✅ dfm ready: clean target, capped extremes, y created. Saved -> amazon_modeling_frame.parquet")

# Price repair & feature build
# Works on: dfm  (keeps shape; no row drop)
# Creates: final_price, price_source, original_price_filled, discount_pct_fixed, ...

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display

# Use dfm from previous step
assert 'dfm' in globals(), "dfm not found. Please run the Target hygiene step first."

dfm = dfm.copy()

# ---- 1) Build final_price (discounted > variant > original) ----
for c in ['discounted_price','price_on_variant','original_price']:
    if c not in dfm.columns:
        dfm[c] = np.nan

cand_disc = dfm['discounted_price'].where(dfm['discounted_price'] > 0)
cand_var  = dfm['price_on_variant'].where(dfm['price_on_variant'] > 0)
cand_orig = dfm['original_price'].where(dfm['original_price'] > 0)

# First non-null of candidates (row-wise)
final_price = cand_disc.combine_first(cand_var).combine_first(cand_orig)

# Price source label
price_source = np.select(
    [
        cand_disc.notna(),
        cand_disc.isna() & cand_var.notna(),
        cand_disc.isna() & cand_var.isna() & cand_orig.notna()
    ],
    ['discounted', 'variant', 'original'],
    default='none'
)

dfm['final_price']  = final_price
dfm['price_source'] = pd.Categorical(price_source, categories=['discounted','variant','original','none'])

# ---- 2) Fill original_price when safely inferable ----
# If original_price is missing, but variant exists and is > discounted by 5%, treat variant as original list price.
thresh = 1.05
orig_filled = dfm['original_price'].copy()
mask_fill = (
    dfm['original_price'].isna()
    & dfm['discounted_price'].notna()
    & dfm['price_on_variant'].notna()
    & (dfm['price_on_variant'] > dfm['discounted_price'] * thresh)
)
orig_filled[mask_fill] = dfm.loc[mask_fill, 'price_on_variant']
dfm['original_price_filled'] = orig_filled

# ---- 3) Recompute discount % using final_price and original_price_filled ----
disc_mask = dfm['original_price_filled'].notna() & dfm['final_price'].notna()
disc_raw  = (dfm['original_price_filled'] - dfm['final_price']) / dfm['original_price_filled'] * 100
# Only keep sane discounts [0, 95]
disc_fixed = disc_raw.where(disc_mask & (disc_raw >= 0) & (disc_raw <= 95))
dfm['discount_pct_fixed'] = disc_fixed
dfm['discount_pct_capped'] = dfm['discount_pct_fixed'].clip(lower=0, upper=95)

# Buckets (useful for tree models)
bins = [-0.1, 0, 5, 10, 20, 30, 50, 95, 1e6]
labels = ['0','0-5','5-10','10-20','20-30','30-50','50-95','>95?']
try:
    dfm['discount_bucket'] = pd.cut(dfm['discount_pct_fixed'], bins=bins, labels=labels, include_lowest=True)
except Exception:
    dfm['discount_bucket'] = pd.Categorical(['0'] * len(dfm), categories=labels)

# ---- 4) Sanity flags & transforms ----
dfm['final_price_missing'] = dfm['final_price'].isna()
dfm['price_inversion'] = (dfm['final_price'] > dfm['original_price_filled']) & dfm['original_price_filled'].notna()
dfm['used_variant_as_orig'] = mask_fill
dfm['log1p_final_price'] = np.log1p(dfm['final_price'])

# ---- 5) QC report ----
print("=== Price Repair Report ===")
print(f"Rows in dfm: {len(dfm)}")
print("\nPrice source counts:")
display(dfm['price_source'].value_counts(dropna=False).to_frame('rows'))

print("\nFinal price summary:")
display(dfm['final_price'].describe(percentiles=[.01,.05,.5,.9,.95,.99]).to_frame('value'))

print("Discount % (fixed) availability:", int(dfm['discount_pct_fixed'].notna().sum()))
print("Price inversions (final_price > orig_filled):", int(dfm['price_inversion'].sum()))
print("Used variant as original fill:", int(dfm['used_variant_as_orig'].sum()))
print("Missing final_price:", int(dfm['final_price_missing'].sum()))

print("\nSample rows (final_price & discount sanity):")
cols_show = [c for c in [
    'product_title','price_source','discounted_price','price_on_variant','original_price',
    'original_price_filled','final_price','discount_pct_fixed','discount_bucket',
    'purchased_last_month','y'
] if c in dfm.columns]
display(dfm[cols_show].head(8))

# Save
dfm.to_parquet("amazon_modeling_frame_prices.parquet", index=False)
print("\n✅ Saved -> amazon_modeling_frame_prices.parquet (dfm updated with price features)")

# Price imputation (tiered medians) + discount rebuild

In [ ]:
# =========================================
# STEP 5b — Price imputation (tiered medians) + discount rebuild
# Works on: dfm  (from previous steps)
# Creates/updates:
#   final_price (imputed where missing), final_price_imputed (bool),
#   impute_tier (which rule used),
#   discount_pct_fixed / discount_pct_capped / discount_bucket (recomputed),
#   price_inversion (recomputed)
# Saves: amazon_modeling_frame_prices_imputed.parquet
# =========================================
import re, unicodedata, numpy as np, pandas as pd
from IPython.display import display

assert 'dfm' in globals(), "dfm not found. Please run the prior steps first."

dfm = dfm.copy()

# ---------- helper: coarse category from title (if needed) ----------
if 'product_category' in dfm.columns:
    coarse = dfm['product_category'].astype(str).str.strip()
else:
    coarse = pd.Series([""] * len(dfm), index=dfm.index)

def infer_coarse_from_title(s: object) -> str:
    if pd.isna(s): return "Other"
    t = unicodedata.normalize("NFKC", str(s)).lower()
    # simple keyword buckets (non-exclusive; first match wins)
    rules = [
        (r'\b(laptop|notebook|macbook)\b', 'Laptop'),
        (r'\b(headphone|earbud|earphone|tws|ear bud|headset|speaker|soundbar)\b', 'Audio'),
        (r'\b(camera|dslr|mirrorless|webcam|action cam|gopro)\b', 'Camera'),
        (r'\b(phone|iphone|android|smartphone|oneplus|samsung galaxy)\b', 'Mobile'),
        (r'\b(ssd|hard\s*drive|hdd|micro\s*sd|sd\s*card|flash\s*drive|pendrive|pen\s*drive)\b', 'Storage'),
        (r'\b(mouse|keyboard|monitor|router|adapter|charger|cable|hub|dock|power\s*bank)\b', 'Accessory'),
        (r'\b(tv|television|projector)\b', 'TV/Display'),
    ]
    for pat, lab in rules:
        if re.search(pat, t):
            return lab
    return "Other"

if coarse.isna().any() or (coarse == "").any():
    fill_idx = coarse[coarse.isna() | (coarse == "")].index
    coarse.loc[fill_idx] = dfm.loc[fill_idx, 'product_title'].map(infer_coarse_from_title)

dfm['coarse_cat'] = coarse.astype('category')

# ---------- helper: brand extraction from title ----------
STOPWORDS = {'the','a','an','new','latest','2025','portable','wireless','with','for','and','by','from','brand'}
def extract_brand(title: object) -> str:
    if pd.isna(title): return 'unknown'
    s = unicodedata.normalize("NFKC", str(title)).strip()
    # take leading segment before separators
    seg = re.split(r'[\-|–|:|\(|\[|,|/]', s, maxsplit=1)[0]
    # special "Brand: XYZ" style
    m = re.search(r'^\s*(?:brand|manufacturer)\s*[:\-]\s*([A-Za-z0-9\-\+\. ]{2,})', seg, flags=re.I)
    if m:
        seg = m.group(1)
    # first token as brand (keep alnum/plus/dot)
    tok = re.findall(r'[A-Za-z0-9\+\.\-]+', seg)
    if not tok:
        return 'unknown'
    cand = tok[0].lower()
    if cand in STOPWORDS or len(cand) <= 1:
        # try second token
        cand = tok[1].lower() if len(tok) > 1 else 'unknown'
    return cand

dfm['brand_key'] = dfm['product_title'].map(extract_brand).astype('category')

# ---------- observed prices (non-missing final_price) ----------
obs_mask = dfm['final_price'].notna() & (dfm['final_price'] > 0)
obs = dfm.loc[obs_mask, ['final_price','brand_key','coarse_cat']].copy()

# global sanity clip bounds to avoid wild imputations
p1  = float(obs['final_price'].quantile(0.01)) if len(obs) else 1.0
p99 = float(obs['final_price'].quantile(0.99)) if len(obs) else 1000.0
global_med = float(obs['final_price'].median()) if len(obs) else 50.0
low, high = max(0.99, p1), max(p1+1, p99)  # ensure sensible ordering

# tier medians
med_brand_cat = obs.groupby(['brand_key','coarse_cat'])['final_price'].median()
med_brand     = obs.groupby(['brand_key'])['final_price'].median()
med_cat       = obs.groupby(['coarse_cat'])['final_price'].median()

# ---------- impute missing final_price ----------
miss_idx = dfm.index[dfm['final_price'].isna() | (dfm['final_price'] <= 0)]
n_missing_before = len(miss_idx)

if n_missing_before:
    miss = dfm.loc[miss_idx, ['brand_key','coarse_cat']].copy()

    # lookups (vectorized-ish)
    key_bc = list(zip(miss['brand_key'], miss['coarse_cat']))
    v1 = pd.Series([med_brand_cat.get(k, np.nan) for k in key_bc], index=miss.index)
    v2 = miss['brand_key'].map(med_brand)
    v3 = miss['coarse_cat'].map(med_cat)
    v4 = pd.Series(global_med, index=miss.index)

    # choose first available
    imputed_vals = v1.combine_first(v2).combine_first(v3).combine_first(v4)

    # clip to sane bounds
    imputed_vals = imputed_vals.clip(lower=low, upper=high)

    # decide tier used
    tier = np.select(
        [v1.notna(), v1.isna() & v2.notna(), v1.isna() & v2.isna() & v3.notna()],
        ['brand+cat','brand','cat'],
        default='global'
    )

    # write back
    dfm.loc[miss_idx, 'final_price'] = imputed_vals
    dfm.loc[miss_idx, 'final_price_imputed'] = True
    dfm.loc[miss_idx, 'impute_tier'] = pd.Categorical(tier, categories=['brand+cat','brand','cat','global'])
else:
    dfm['final_price_imputed'] = False
    dfm['impute_tier'] = pd.Categorical(['']*len(dfm))

dfm['final_price_imputed'] = dfm['final_price_imputed'].fillna(False).astype(bool)

# ---------- rebuild discount features using original_price_filled ----------
if 'original_price_filled' not in dfm.columns:
    # fallback to original_price if previous step wasn't run
    dfm['original_price_filled'] = dfm.get('original_price', pd.Series(index=dfm.index, dtype='float64'))

disc_mask = dfm['original_price_filled'].notna() & dfm['final_price'].notna()
disc_raw  = (dfm['original_price_filled'] - dfm['final_price']) / dfm['original_price_filled'] * 100
disc_fixed = disc_raw.where(disc_mask & (disc_raw >= 0) & (disc_raw <= 95))
dfm['discount_pct_fixed']  = disc_fixed
dfm['discount_pct_capped'] = dfm['discount_pct_fixed'].clip(lower=0, upper=95)

bins   = [-0.1, 0, 5, 10, 20, 30, 50, 95, 1e6]
labels = ['0','0-5','5-10','10-20','20-30','30-50','50-95','>95?']
try:
    dfm['discount_bucket'] = pd.cut(dfm['discount_pct_fixed'], bins=bins, labels=labels, include_lowest=True)
except Exception:
    dfm['discount_bucket'] = pd.Categorical(['0']*len(dfm), categories=labels)

# recompute inversion flag with imputed prices
dfm['price_inversion'] = (dfm['final_price'] > dfm['original_price_filled']) & dfm['original_price_filled'].notna()

# log price
dfm['log1p_final_price'] = np.log1p(dfm['final_price'])

# ---------- QC report ----------
n_missing_after = int(dfm['final_price'].isna().sum() + (dfm['final_price'] <= 0).sum())
print("=== Price Imputation Report ===")
print(f"Observed prices before: {int(obs_mask.sum())} / {len(dfm)}")
print(f"Missing/invalid before: {n_missing_before}")
print(f"Missing/invalid after:  {n_missing_after}")

if n_missing_before:
    print("\nImpute tiers used (missing rows only):")
    display(dfm.loc[miss_idx, 'impute_tier'].value_counts(dropna=False).to_frame('rows'))

print("\nFinal_price summary (post-imputation):")
display(dfm['final_price'].describe(percentiles=[.01,.05,.5,.9,.95,.99]).to_frame('value'))

print("Discount % (fixed) availability:", int(dfm['discount_pct_fixed'].notna().sum()))
print("Price inversions:", int(dfm['price_inversion'].sum()))
print("Imputed rows:", int(dfm['final_price_imputed'].sum()))

print("\nSample of imputed rows:")
cols_show = [c for c in [
    'product_title','brand_key','coarse_cat','impute_tier',
    'final_price','original_price','price_on_variant','discounted_price',
    'original_price_filled','discount_pct_fixed','discount_bucket',
    'purchased_last_month','y'
] if c in dfm.columns]
display(dfm[dfm['final_price_imputed']].head(8)[cols_show])

# Save
dfm.to_parquet("amazon_modeling_frame_prices_imputed.parquet", index=False)
print("\n✅ Saved -> amazon_modeling_frame_prices_imputed.parquet (dfm updated with imputed prices)")

# Boolean salvage from text/URLs

In [ ]:
import re, unicodedata, numpy as np, pandas as pd
from IPython.display import display

assert 'dfm' in globals(), "dfm not found. Please run prior steps first."
dfm = dfm.copy()

# --- helpers ---
def norm_text(s):
    if pd.isna(s): return ""
    return unicodedata.normalize("NFKC", str(s)).lower()

def col_contains(sr: pd.Series, pattern: re.Pattern) -> pd.Series:
    return sr.astype(str).str.contains(pattern, na=False)

def any_contains(df: pd.DataFrame, cols, pattern: re.Pattern) -> pd.Series:
    m = pd.Series(False, index=df.index)
    for c in cols:
        if c in df.columns:
            m = m | col_contains(df[c].astype(str), pattern)
    return m

def boolish_from_obj(x) -> bool:
    if isinstance(x, bool): return x
    s = str(x).strip().lower()
    if s in {'true','1','yes','y','sponsored','best seller','bestseller','coupon','has coupon'}: return True
    if s in {'false','0','no','n'}: return False
    return False

# --- original booleans (coerced safely) ---
orig_sponsored = dfm.get('is_sponsored', pd.Series(index=dfm.index)).map(boolish_from_obj).fillna(False)
orig_best      = dfm.get('is_best_seller', pd.Series(index=dfm.index)).map(boolish_from_obj).fillna(False)
orig_coupon    = dfm.get('has_coupon', pd.Series(index=dfm.index)).map(boolish_from_obj).fillna(False)

# --- patterns (conservative but useful) ---
pat_sponsored = re.compile(r'(?:\bsspa\b|/sspa/click|sponsored|adid=|aax-)', re.I)
pat_coupon    = re.compile(r'(?:coupon|apply coupon|with coupon|save extra|clip(?:pable)? coupon|coupon available)', re.I)
pat_best      = re.compile(r'(?:#?1\s*best\s*seller|best\s*seller)', re.I)
# card phrases that often imply lack of buy box / not a true product card
pat_no_offer  = re.compile(r'(?:no featured offers available|more buying choices|list\s*price:|typical:)', re.I)

scan_cols = [c for c in ['product_title','bought_in_last_month','sustainability_tags','product_page_url'] if c in dfm.columns]

heur_sponsored = any_contains(dfm, scan_cols, pat_sponsored)
heur_coupon    = any_contains(dfm, scan_cols, pat_coupon)
heur_best      = any_contains(dfm, scan_cols, pat_best)
no_offer_flag  = any_contains(dfm, ['bought_in_last_month'], pat_no_offer)

# --- fixed booleans (OR original with heuristic) ---
dfm['is_sponsored_fixed']  = (orig_sponsored | heur_sponsored).astype(bool)
dfm['has_coupon_fixed']    = (orig_coupon    | heur_coupon).astype(bool)
dfm['is_best_seller_fixed']= (orig_best      | heur_best).astype(bool)
dfm['no_featured_offer_flag'] = no_offer_flag.astype(bool)

# --- report ---
def cnt_true(sr): return int(sr.fillna(False).sum())

print("=== Boolean Salvage Report ===")
print(f"Rows: {len(dfm)}")

print("\nOriginal -> After (True counts)")
spon_before, spon_after = cnt_true(orig_sponsored), cnt_true(dfm['is_sponsored_fixed'])
best_before, best_after = cnt_true(orig_best),      cnt_true(dfm['is_best_seller_fixed'])
coup_before, coup_after = cnt_true(orig_coupon),    cnt_true(dfm['has_coupon_fixed'])

print(f"is_sponsored: {spon_before} -> {spon_after}  (+{spon_after - spon_before})")
print(f"is_best_seller: {best_before} -> {best_after}  (+{best_after - best_before})")
print(f"has_coupon: {coup_before} -> {coup_after}  (+{coup_after - coup_before})")
print(f"no_featured_offer_flag (heuristic only): {cnt_true(dfm['no_featured_offer_flag'])}")

# quick cross-tab sanity
print("\nSponsored by source (URL/text) among newly-flagged:")
new_spon = dfm[(~orig_sponsored) & (dfm['is_sponsored_fixed'])]
if len(new_spon):
    cols_show = [c for c in ['product_title','product_page_url','bought_in_last_month'] if c in dfm.columns]
    display(new_spon[cols_show].head(5))
else:
    print("[None newly flagged]")

print("\nCoupon examples (newly-flagged):")
new_coup = dfm[(~orig_coupon) & (dfm['has_coupon_fixed'])]
if len(new_coup):
    cols_show = [c for c in ['product_title','bought_in_last_month','product_page_url'] if c in dfm.columns]
    display(new_coup[cols_show].head(5))
else:
    print("[None newly flagged]")

# Save
dfm.to_parquet("amazon_modeling_frame_booleans.parquet", index=False)
print("\n✅ Saved -> amazon_modeling_frame_booleans.parquet (dfm updated with *_fixed flags)")

# Time-based CV scaffold (forward-chaining)

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display

assert 'dfm' in globals(), "dfm not found. Please run previous steps first."
assert 'collected_at_dt' in dfm.columns, "dfm must have 'collected_at_dt'."
if 'y' not in dfm.columns and 'purchased_last_month' in dfm.columns:
    dfm['y'] = np.log1p(dfm['purchased_last_month'])

FOLDS = 5  # change to 4/6 etc. as you like

dfm = dfm.copy()
dfm = dfm[dfm['collected_at_dt'].notna()].copy()  # drop rows with no time (should be none)
dfm.sort_values('collected_at_dt', inplace=True)
dfm.reset_index(drop=True, inplace=True)

# Convert timestamps to int for stable binning
ts = dfm['collected_at_dt'].astype('int64')  # ns since epoch

# Contiguous time bins with near-equal sizes
try:
    dfm['time_bin'] = pd.qcut(ts, q=FOLDS, labels=False, duplicates='drop')
except Exception:
    # fallback: manual quantile edges
    qs = np.linspace(0, 1, FOLDS + 1)
    edges = np.quantile(ts, qs)
    # tiny nudges to ensure strictly increasing edges
    for i in range(1, len(edges)):
        if edges[i] <= edges[i-1]:
            edges[i] = edges[i-1] + 1
    dfm['time_bin'] = np.digitize(ts, edges[1:-1], right=True)

dfm['time_bin'] = dfm['time_bin'].astype(int)
bins_sorted = sorted(dfm['time_bin'].unique())

# Build expanding folds: for each bin k>=1, train on bins < k, validate on bin == k
fold_ids = []
dfm['fold'] = -1
fold_meta = []
fold_num = 0
for k in bins_sorted:
    if k == bins_sorted[0]:
        # skip first bin (no past to train on)
        continue
    val_idx = dfm.index[dfm['time_bin'] == k].to_numpy()
    trn_idx = dfm.index[dfm['time_bin'] <  k].to_numpy()
    if len(trn_idx) == 0 or len(val_idx) == 0:
        continue
    # mark fold on dfm (validation rows)
    dfm.loc[val_idx, 'fold'] = fold_num
    # meta report
    trn_min, trn_max = dfm.loc[trn_idx, 'collected_at_dt'].min(), dfm.loc[trn_idx, 'collected_at_dt'].max()
    val_min, val_max = dfm.loc[val_idx, 'collected_at_dt'].min(), dfm.loc[val_idx, 'collected_at_dt'].max()
    fold_meta.append({
        'fold': fold_num,
        'train_rows': len(trn_idx),
        'valid_rows': len(val_idx),
        'train_start': trn_min, 'train_end': trn_max,
        'valid_start': val_min, 'valid_end': val_max
    })
    fold_ids.append((trn_idx, val_idx))
    fold_num += 1

# Save indices
npz_dict = {f'fold{i}_train_idx': tr for i,(tr,va) in enumerate(fold_ids)}
npz_dict.update({f'fold{i}_valid_idx': va for i,(tr,va) in enumerate(fold_ids)})
np.savez_compressed('cv_splits_time_expanding.npz', **npz_dict)

# Save meta
meta_df = pd.DataFrame(fold_meta)
meta_df.to_csv('cv_folds_meta.csv', index=False)

# Quick report
print("=== Time-based CV (expanding) ===")
print(f"Rows in dfm (with datetime): {len(dfm)}")
overall_min, overall_max = dfm['collected_at_dt'].min(), dfm['collected_at_dt'].max()
print(f"Date range: {overall_min} -> {overall_max}")
print("\nTime bin counts:")
display(dfm['time_bin'].value_counts().sort_index().to_frame('rows'))

print("\nFolds created:", len(fold_ids))
display(meta_df)

# y-bin sanity in each validation fold (if cv_strat5 exists)
if 'cv_strat5' in dfm.columns:
    print("\nValidation y-bin distribution per fold:")
    for i, (_, va) in enumerate(fold_ids):
        vc = dfm.loc[va, 'cv_strat5'].value_counts().sort_index()
        print(f"Fold {i}:", dict(vc))

# Save dfm with fold columns
dfm.to_parquet('amazon_modeling_frame_cv.parquet', index=False)
print("\n✅ Saved:")
print(" - cv_splits_time_expanding.npz (train/valid indices)")
print(" - cv_folds_meta.csv (fold date ranges & sizes)")
print(" - amazon_modeling_frame_cv.parquet (dfm with time_bin & fold)")

# FULL EDA MINI-SUITE (robust, saves figures)

In [ ]:
import os, re, gc, math, warnings, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 140
OUT = Path("eda"); OUT.mkdir(exist_ok=True)

# ---------- 0) Choose frame ----------
def try_load_raw():
    p = "/kaggle/input/amazon-products-sales-dataset-42k-items-2025/amazon_products_sales_data_uncleaned.csv"
    return pd.read_csv(p, low_memory=False)

if 'dfm' in globals():
    base_name, df_used = "dfm (modeling)", dfm.copy()
elif 'dfd' in globals():
    base_name, df_used = "dfd (deduped)", dfd.copy()
elif 'df' in globals():
    base_name, df_used = "df (raw)", df.copy()
else:
    base_name, df_used = "df (raw, loaded)", try_load_raw()

print(f"Using frame: {base_name} | shape={df_used.shape}")

# ---------- 1) Helpers ----------
def ensure_col(df, name, default=np.nan):
    if name not in df.columns: df[name] = default
    return name

def exists(df, cols): return all(c in df.columns for c in cols)

def infer_coarse_from_title(s: object) -> str:
    if pd.isna(s): return "Other"
    t = unicodedata.normalize("NFKC", str(s)).lower()
    rules = [
        (r'\b(laptop|notebook|macbook)\b', 'Laptop'),
        (r'\b(headphone|earbud|earphone|tws|headset|speaker|soundbar)\b', 'Audio'),
        (r'\b(camera|dslr|mirrorless|webcam|action cam|gopro)\b', 'Camera'),
        (r'\b(phone|iphone|android|smartphone|oneplus|samsung galaxy)\b', 'Mobile'),
        (r'\b(ssd|hard\s*drive|hdd|micro\s*sd|sd\s*card|flash\s*drive|pendrive|pen\s*drive)\b', 'Storage'),
        (r'\b(mouse|keyboard|monitor|router|adapter|charger|cable|hub|dock|power\s*bank)\b', 'Accessory'),
        (r'\b(tv|television|projector)\b', 'TV/Display'),
    ]
    for pat, lab in rules:
        if re.search(pat, t): return lab
    return "Other"

STOPWORDS = {'the','a','an','new','latest','2025','portable','wireless','with','for','and','by','from','brand'}
def extract_brand(title: object) -> str:
    if pd.isna(title): return 'unknown'
    s = unicodedata.normalize("NFKC", str(title)).strip()
    seg = re.split(r'[\-|–|:|\(|\[|,|/]', s, maxsplit=1)[0]
    m = re.search(r'^\s*(?:brand|manufacturer)\s*[:\-]\s*([A-Za-z0-9\-\+\. ]{2,})', seg, flags=re.I)
    if m: seg = m.group(1)
    toks = re.findall(r'[A-Za-z0-9\+\.\-]+', seg)
    cand = (toks[0] if toks else 'unknown').lower()
    if cand in STOPWORDS or len(cand) <= 1:
        cand = toks[1].lower() if len(toks) > 1 else 'unknown'
    return cand

def savefig(fig, name):
    p = OUT / f"{name}.png"
    fig.savefig(p, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {p}")

# Soft-copy to avoid mutating globals
df = df_used.copy()

# ---------- 2) Light feature prep for EDA ----------
# Canonical names if raw
if 'product_title' not in df.columns and 'title' in df.columns:
    df['product_title'] = df['title']

# Category/Brand
if 'coarse_cat' not in df.columns and 'product_title' in df.columns:
    df['coarse_cat'] = df['product_title'].map(infer_coarse_from_title)
if 'brand_key' not in df.columns and 'product_title' in df.columns:
    df['brand_key'] = df['product_title'].map(extract_brand)

# Target columns
target_cols = [c for c in ['purchased_last_month','target_log1p','y'] if c in df.columns]
target_name = target_cols[0] if target_cols else None

# Price columns
for c in ['final_price','discounted_price','price_on_variant','original_price','original_price_filled','discount_pct_capped','discount_percentage']:
    ensure_col(df, c)

# Rating / reviews transforms
if 'log1p_total_reviews' not in df.columns and 'total_reviews' in df.columns:
    df['log1p_total_reviews'] = np.log1p(pd.to_numeric(df['total_reviews'], errors='coerce'))

# Time
if 'collected_at_dt' not in df.columns and 'collected_at' in df.columns:
    df['collected_at_dt'] = pd.to_datetime(df['collected_at'], errors='coerce', infer_datetime_format=True)

# Title lengths
if 'product_title' in df.columns:
    df['title_chars'] = df['product_title'].astype(str).str.len()
    df['title_words'] = df['product_title'].astype(str).str.split().map(len)

# ---------- 3) High-level info ----------
print("\n=== Head (5) ===")
display(df.head(5))
print("\n=== Dtypes ===")
print(df.dtypes)

# ---------- 4) Missingness (top 25) ----------
miss = df.isna().sum().sort_values(ascending=False)
miss_df = pd.DataFrame({'missing': miss, 'missing_pct': (miss/len(df)*100).round(2)}).head(25)
display(miss_df)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(y=miss_df.index, x=miss_df['missing_pct'], color="#4C78A8", ax=ax)
ax.set(title="Top Missingness (%)", xlabel="% missing", ylabel="")
savefig(fig, "01_missingness_top")

# ---------- 5) Target distribution ----------
if target_name is not None:
    s = df[target_name].dropna()
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    sns.histplot(s, bins=50, ax=axs[0], color="#4C78A8")
    axs[0].set_title(f"{target_name} (linear)")
    sns.histplot(np.log1p(s), bins=50, ax=axs[1], color="#F58518")
    axs[1].set_title(f"log1p({target_name})")
    savefig(fig, "02_target_distribution")
    print(f"{target_name} summary:")
    display(s.describe(percentiles=[.01,.05,.5,.9,.95,.99]).to_frame("value"))
else:
    print("\n[Note] No parsed target column found for distribution plot.")

# ---------- 6) Price distributions ----------
num_price_cols = [c for c in ['final_price','discounted_price','original_price','price_on_variant'] if c in df.columns]
if num_price_cols:
    fig, axs = plt.subplots(2, 2, figsize=(10, 7))
    axs = axs.flatten()
    for i, c in enumerate(num_price_cols[:4]):
        sns.histplot(df[c].dropna(), bins=60, ax=axs[i], color="#54A24B")
        axs[i].set_title(c)
    savefig(fig, "03_price_distributions")

# ---------- 7) Discount %
if 'discount_pct_capped' in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df['discount_pct_capped'].dropna(), bins=40, color="#E45756", ax=ax)
    ax.set_title("Discount % (capped)")
    savefig(fig, "04_discount_pct")

# ---------- 8) Target vs Price/Discount/Rating ----------
if target_name is not None:
    # Price vs target (log scales for both)
    if 'final_price' in df.columns and df['final_price'].notna().any():
        sub = df[[target_name,'final_price']].dropna().copy()
        sub['logp'] = np.log1p(sub['final_price'])
        sub['logy'] = np.log1p(sub[target_name])
        fig, ax = plt.subplots(figsize=(6.5, 5))
        sns.kdeplot(data=sub, x='logp', y='logy', fill=True, cmap="mako", thresh=0.05, ax=ax)
        ax.set(title="log(price) vs log(target)", xlabel="log1p(final_price)", ylabel=f"log1p({target_name})")
        savefig(fig, "05_logprice_vs_logtarget")

    # Discount buckets vs target
    for bcol in ['discount_bucket']:
        if bcol in df.columns:
            tmp = df[[bcol, target_name]].dropna().copy()
            if tmp[bcol].dtype.name != 'category':
                tmp[bcol] = tmp[bcol].astype(str)
            fig, ax = plt.subplots(figsize=(8,4))
            order = None
            try:
                order = ['0','0-5','5-10','10-20','20-30','30-50','50-95','>95?']
            except Exception:
                pass
            sns.boxplot(data=tmp, x=bcol, y=np.log1p(tmp[target_name]), order=order, ax=ax, color="#72B7B2")
            ax.set_title("Discount bucket vs log target")
            ax.set_ylabel(f"log1p({target_name})")
            savefig(fig, "06_discount_bucket_vs_target")

    # Rating / reviews effects
    if 'product_rating' in df.columns:
        fig, ax = plt.subplots(figsize=(6.5,4))
        sns.scatterplot(data=df, x='product_rating', y=np.log1p(df[target_name]), alpha=0.2, ax=ax, color="#FF9DA7")
        ax.set_title("Rating vs log target")
        ax.set_ylabel(f"log1p({target_name})")
        savefig(fig, "07_rating_vs_target")

    if 'log1p_total_reviews' in df.columns:
        fig, ax = plt.subplots(figsize=(6.5,4))
        sns.scatterplot(data=df, x='log1p_total_reviews', y=np.log1p(df[target_name]), alpha=0.2, ax=ax, color="#E2CF70")
        ax.set_title("log reviews vs log target")
        ax.set_ylabel(f"log1p({target_name})")
        savefig(fig, "08_reviews_vs_target")

# ---------- 9) Categories & Brands ----------
if 'coarse_cat' in df.columns:
    vc = df['coarse_cat'].astype(str).value_counts().head(12)
    fig, ax = plt.subplots(figsize=(7,4))
    sns.barplot(x=vc.values, y=vc.index, color="#4C78A8", ax=ax)
    ax.set_title("Top coarse categories")
    savefig(fig, "09_top_coarse_categories")
if 'brand_key' in df.columns:
    vb = df['brand_key'].astype(str).value_counts().head(20)
    fig, ax = plt.subplots(figsize=(7,6))
    sns.barplot(x=vb.values, y=vb.index, color="#F58518", ax=ax)
    ax.set_title("Top brands (approx. from title)")
    savefig(fig, "10_top_brands")

# ---------- 10) Sponsored / Best Seller effects ----------
for flag in ['is_sponsored','is_sponsored_fixed','is_best_seller','is_best_seller_fixed','has_coupon_fixed']:
    if flag in df.columns and target_name is not None:
        fig, ax = plt.subplots(figsize=(5.8,4))
        tmp = df[[flag, target_name]].dropna()
        if not tmp.empty:
            sns.boxplot(data=tmp, x=flag, y=np.log1p(tmp[target_name]), ax=ax, color="#54A24B")
            ax.set_title(f"{flag} vs log target")
            ax.set_ylabel(f"log1p({target_name})")
            savefig(fig, f"11_{flag}_vs_target")

# ---------- 11) Time series (if available) ----------
if 'collected_at_dt' in df.columns and df['collected_at_dt'].notna().any():
    df['date_hour'] = df['collected_at_dt'].dt.floor('H')
    fig, ax = plt.subplots(figsize=(8,4))
    df.groupby('date_hour').size().plot(ax=ax, color="#4C78A8")
    ax.set_title("Rows per hour")
    ax.set_ylabel("count")
    savefig(fig, "12_rows_per_hour")

    if target_name is not None:
        fig, ax = plt.subplots(figsize=(8,4))
        grp = df.dropna(subset=[target_name,'date_hour']).groupby('date_hour')[target_name].median()
        grp.apply(np.log1p).plot(ax=ax, color="#E45756")
        ax.set_title("Median log target by hour")
        ax.set_ylabel(f"log1p({target_name})")
        savefig(fig, "13_median_target_by_hour")

# ---------- 12) Correlation heatmap (numerics) ----------
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if df[c].notna().sum() > 0]
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    fig, ax = plt.subplots(figsize=(min(12, 0.55*len(num_cols)+3), min(10, 0.55*len(num_cols)+3)))
    sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlation heatmap (numeric features)")
    savefig(fig, "14_corr_heatmap")

# ---------- 13) Duplicates snapshot if raw+dedup are available ----------
if 'df' in globals() and 'dfd' in globals():
    raw_n = len(df)
    dedup_n = len(dfd)
    print(f"\nDuplicates snapshot (from globals): raw={len(df)}, dedup={len(dfd)} (Δ {len(df)-len(dfd)})")

print("\n✅ EDA complete. Check the 'eda/' folder for PNGs.")

# SHOW EDA FIGURES INLINE (gallery)

In [ ]:
from pathlib import Path
from IPython.display import display, HTML
import base64, os

OUT = Path("eda")
if not OUT.exists():
    raise FileNotFoundError("No 'eda/' folder found. Run the EDA cell first to generate plots.")

pngs = sorted(OUT.glob("*.png"))
if not pngs:
    raise FileNotFoundError("No PNGs found in 'eda/'. Re-run the EDA cell to create them.")

def img_tag(path, width=420):
    b64 = base64.b64encode(path.read_bytes()).decode("ascii")
    name = os.path.splitext(path.name)[0].replace("_", " ")
    return f'''
    <figure style="margin:8px">
      <img src="data:image/png;base64,{b64}" width="{width}" style="border:1px solid #eee;border-radius:6px;">
      <figcaption style="text-align:center;font-size:12px;color:#555">{name}</figcaption>
    </figure>'''

html = '<div style="display:flex;flex-wrap:wrap;gap:8px;align-items:flex-start">'
for p in pngs:
    html += img_tag(p, width=420)
html += '</div>'
display(HTML(html))

# Quick baseline: LightGBM + TF-IDF(title) with time-based CV

In [ ]:
import os, gc, numpy as np, pandas as pd, scipy.sparse as sp
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
import lightgbm as lgb
from IPython.display import display

SEED = 42
np.random.seed(SEED)

# -----------------------------
# Load dfm if needed
# -----------------------------
if 'dfm' not in globals():
    dfm = pd.read_parquet('amazon_modeling_frame_cv.parquet')

# Ensure required columns
assert 'y' in dfm.columns, "Missing 'y' (log1p target). Please run earlier steps."
assert 'fold' in dfm.columns, "Missing 'fold' column. Please run CV scaffold step."
assert 'product_title' in dfm.columns, "Missing 'product_title'."

# Optionally load fold indices file (preferred). Else rebuild from 'fold'.
fold_ids = []
if os.path.exists('cv_splits_time_expanding.npz'):
    npz = np.load('cv_splits_time_expanding.npz', allow_pickle=True)
    i = 0
    while f'fold{i}_train_idx' in npz and f'fold{i}_valid_idx' in npz:
        tr = npz[f'fold{i}_train_idx']
        va = npz[f'fold{i}_valid_idx']
        if len(tr) and len(va):
            fold_ids.append((tr, va))
        i += 1
else:
    # fallback from 'fold' bins: for each k>=0, train on rows with time_bin < bin(k)
    bins = sorted(dfm.loc[dfm['fold'] >= 0, 'time_bin'].unique().tolist())
    for i, k in enumerate(bins):
        va = dfm.index[dfm['fold'] == i].to_numpy()
        tr = dfm.index[dfm['time_bin'] < k].to_numpy()
        if len(tr) and len(va):
            fold_ids.append((tr, va))

assert len(fold_ids) > 0, "No folds found."

# -----------------------------
# Feature sets (tabular + text)
# -----------------------------
# Build a local copy to avoid mutating dfm
dfX = dfm.copy()

# Booleans -> uint8 for safe numeric handling
bool_cols = [c for c in ['is_sponsored_fixed','is_best_seller_fixed','has_coupon_fixed',
                         'buy_box_availability','has_discount','final_price_imputed',
                         'used_variant_as_orig','no_featured_offer_flag']
             if c in dfX.columns]
for c in bool_cols:
    dfX[c] = dfX[c].fillna(False).astype('uint8')

# Helpful numeric features (avoid target-derived ones!)
num_cols = [c for c in [
    'product_rating',               # 0..5
    'total_reviews',                # count (we'll impute)
    'log1p_final_price',            # stabilized price signal
    'final_price',                  # raw price
    'discount_pct_capped',          # 0..95
] if c in dfX.columns]

# Simple transform: log reviews to tame the tail (create a new column locally)
if 'total_reviews' in num_cols:
    dfX['log1p_total_reviews'] = np.log1p(dfX['total_reviews'].fillna(0))
    # replace raw reviews with log version to avoid duplicating signal
    num_cols = [c for c in num_cols if c != 'total_reviews'] + ['log1p_total_reviews']

cat_cols = [c for c in ['price_source','coarse_cat','brand_key'] if c in dfX.columns]
text_col = 'product_title'

# ColumnTransformer: numeric impute, onehot for cats, TF-IDF for text
preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse=True), cat_cols),
        ('txt', TfidfVectorizer(
            sublinear_tf=True,
            ngram_range=(1, 2),
            min_df=5,
            max_features=30000,
            dtype=np.float32
        ), text_col),
    ],
    remainder='drop',
    sparse_threshold=1.0
)

# -----------------------------
# LightGBM params
# -----------------------------
lgb_params = {
    'objective': 'rmse',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 255,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'min_data_in_leaf': 20,
    'lambda_l1': 0.1,
    'lambda_l2': 0.5,
    'max_bin': 255,
    'verbose': -1,
    'seed': SEED,
    'num_threads': -1,
    'force_col_wise': True,   # better with many sparse features
}

# -----------------------------
# CV training loop
# -----------------------------
y = dfX['y'].to_numpy()
n = len(dfX)
oof = np.full(n, np.nan, dtype=np.float32)

feature_gain = None  # aggregated importance
fold_summaries = []

for i, (tr_idx, va_idx) in enumerate(fold_ids):
    print(f"\n==== Fold {i} ({len(tr_idx)} train, {len(va_idx)} valid) ====")
    X_tr = preprocess.fit_transform(dfX.iloc[tr_idx])
    X_va = preprocess.transform(dfX.iloc[va_idx])
    y_tr, y_va = y[tr_idx], y[va_idx]

    # feature names (for importances)
    try:
        feat_names = preprocess.get_feature_names_out()
    except Exception:
        # fallback (unnamed)
        feat_names = np.array([f'f{j}' for j in range(X_tr.shape[1])])

    # LightGBM train API (to preserve feature names)
    dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=list(feat_names), free_raw_data=False)
    dvalid = lgb.Dataset(X_va, label=y_va, feature_name=list(feat_names), free_raw_data=False)

    model = lgb.train(
        params=lgb_params,
        train_set=dtrain,
        num_boost_round=3000,
        valid_sets=[dvalid],
        valid_names=['valid'],
        callbacks=[lgb.early_stopping(stopping_rounds=200), lgb.log_evaluation(200)]
    )

    pred_va = model.predict(X_va, num_iteration=model.best_iteration)
    oof[va_idx] = pred_va

    rmse_log = mean_squared_error(y_va, pred_va, squared=False)
    rmse_units = mean_squared_error(np.expm1(y_va), np.expm1(pred_va), squared=False)

    print(f"Fold {i} — RMSE(log): {rmse_log:.4f} | RMSE(units): {rmse_units:,.1f}")

    # importances
    imp = pd.DataFrame({
        'feature': model.feature_name(),
        'gain': model.feature_importance(importance_type='gain')
    })
    imp['fold'] = i

    feature_gain = imp if feature_gain is None else pd.concat([feature_gain, imp], axis=0, ignore_index=True)

    # save model
    model.save_model(f'lgbm_model_fold{i}.txt', num_iteration=model.best_iteration)

    # free memory
    del X_tr, X_va, dtrain, dvalid, model
    gc.collect()

# -----------------------------
# OOF summary
# -----------------------------
mask_oof = ~np.isnan(oof)
oof_rmse_log = mean_squared_error(y[mask_oof], oof[mask_oof], squared=False)
oof_rmse_units = mean_squared_error(np.expm1(y[mask_oof]), np.expm1(oof[mask_oof]), squared=False)

print("\n==== OOF performance ====")
print(f"OOF RMSE(log):   {oof_rmse_log:.4f}")
print(f"OOF RMSE(units): {oof_rmse_units:,.1f}")

# -----------------------------
# Save OOF preds
# -----------------------------
id_col = 'asin_key' if 'asin_key' in dfX.columns else ('product_page_url' if 'product_page_url' in dfX.columns else None)
oof_df = pd.DataFrame({
    'row_id': dfX.index,
    'id': dfX[id_col] if id_col else np.nan,
    'y_true': y,
    'y_pred': oof,
    'fold': dfX['fold']
})
oof_df.to_csv('oof_preds.csv', index=False)
print("\n✅ Saved: oof_preds.csv")

# -----------------------------
# Aggregate and show top features
# -----------------------------
if feature_gain is not None and len(feature_gain):
    agg = (
        feature_gain.groupby('feature', as_index=False)['gain']
        .mean()
        .sort_values('gain', ascending=False)
    )
    top30 = agg.head(30)
    top30.to_csv('feature_importance_top30.csv', index=False)
    print("\nTop 30 features by mean gain across folds:")
    display(top30)
    print("✅ Saved: feature_importance_top30.csv")
else:
    print("\n[Note] No feature importances collected.")

print("\n🎯 Baseline complete.")

# TF-IDF upgrade (word 1–2g + char_wb 3–5g, 100k vocab) + LightGBM

In [ ]:
import os, gc, numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from IPython.display import display

SEED = 42
np.random.seed(SEED)

# -----------------------------
# Load dfm & folds
# -----------------------------
if 'dfm' not in globals():
    dfm = pd.read_parquet('amazon_modeling_frame_cv.parquet')
assert {'y','fold','product_title'}.issubset(dfm.columns)

fold_ids = []
if os.path.exists('cv_splits_time_expanding.npz'):
    npz = np.load('cv_splits_time_expanding.npz', allow_pickle=True)
    i = 0
    while f'fold{i}_train_idx' in npz and f'fold{i}_valid_idx' in npz:
        tr = npz[f'fold{i}_train_idx']; va = npz[f'fold{i}_valid_idx']
        if len(tr) and len(va): fold_ids.append((tr, va))
        i += 1
else:
    bins = sorted(dfm.loc[dfm['fold'] >= 0, 'time_bin'].unique().tolist())
    for i, k in enumerate(bins):
        va = dfm.index[dfm['fold'] == i].to_numpy()
        tr = dfm.index[dfm['time_bin'] < k].to_numpy()
        if len(tr) and len(va): fold_ids.append((tr, va))
assert len(fold_ids) > 0

# -----------------------------
# Feature prep
# -----------------------------
X = dfm.copy()

# Booleans -> uint8
bool_cols = [c for c in [
    'is_sponsored_fixed','is_best_seller_fixed','has_coupon_fixed',
    'buy_box_availability','has_discount','final_price_imputed',
    'used_variant_as_orig','no_featured_offer_flag'
] if c in X.columns]
for c in bool_cols:
    X[c] = X[c].fillna(False).astype('uint8')

# Numeric features
num_cols = [c for c in [
    'product_rating',
    'log1p_total_reviews' if 'log1p_total_reviews' in X.columns else None,
    'log1p_final_price','final_price','discount_pct_capped'
] if c and c in X.columns]

# Categorical features (OHE)
cat_cols = [c for c in ['price_source','coarse_cat','brand_key','discount_bucket'] if c in X.columns]

text_col = 'product_title'

preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse=True), cat_cols),
        ('txt_word', TfidfVectorizer(
            analyzer='word', ngram_range=(1,2), min_df=3, max_df=0.95,
            max_features=50_000, stop_words='english',
            sublinear_tf=True, strip_accents='unicode', dtype=np.float32
        ), text_col),
        ('txt_char', TfidfVectorizer(
            analyzer='char_wb', ngram_range=(3,5), min_df=3, max_df=1.0,
            max_features=50_000, sublinear_tf=True, strip_accents='unicode',
            dtype=np.float32
        ), text_col),
    ],
    remainder='drop', sparse_threshold=1.0
)

# LightGBM params (same as before)
lgb_params = {
    'objective':'rmse','metric':'rmse','learning_rate':0.05,'num_leaves':255,
    'feature_fraction':0.75,'bagging_fraction':0.75,'bagging_freq':1,
    'min_data_in_leaf':25,'lambda_l1':0.5,'lambda_l2':1.0,'max_bin':255,
    'verbose':-1,'seed':SEED,'num_threads':-1,'force_col_wise':True
}

y = X['y'].to_numpy()
n = len(X)
oof = np.full(n, np.nan, dtype=np.float32)
imp_all = []

# -----------------------------
# CV loop (no feature names passed)
# -----------------------------
for i, (tr_idx, va_idx) in enumerate(fold_ids):
    print(f"\n==== TFIDF-Upgrade (no-names) Fold {i} ({len(tr_idx)} train, {len(va_idx)} valid) ====")
    X_tr = preprocess.fit_transform(X.iloc[tr_idx])
    X_va = preprocess.transform(X.iloc[va_idx])
    y_tr, y_va = y[tr_idx], y[va_idx]

    # Create datasets WITHOUT feature_name
    dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
    dvalid = lgb.Dataset(X_va, label=y_va, free_raw_data=False)

    model = lgb.train(
        params=lgb_params,
        train_set=dtrain,
        num_boost_round=5000,
        valid_sets=[dvalid],
        valid_names=['valid'],
        callbacks=[lgb.early_stopping(stopping_rounds=300), lgb.log_evaluation(200)]
    )

    pred_va = model.predict(X_va, num_iteration=model.best_iteration)
    oof[va_idx] = pred_va

    rmse_log = mean_squared_error(y_va, pred_va, squared=False)
    rmse_units = mean_squared_error(np.expm1(y_va), np.expm1(pred_va), squared=False)
    print(f"Fold {i} — RMSE(log): {rmse_log:.4f} | RMSE(units): {rmse_units:,.1f}")

    # Importances by index (names not available)
    imp = pd.DataFrame({
        'feature_idx': np.arange(model.num_feature()),
        'gain': model.feature_importance(importance_type='gain')
    })
    imp['fold'] = i
    imp_all.append(imp)

    model.save_model(f'lgbm_tfidf_union_nonames_fold{i}.txt', num_iteration=model.best_iteration)

    del X_tr, X_va, dtrain, dvalid, model
    gc.collect()

# -----------------------------
# OOF metrics
# -----------------------------
mask = ~np.isnan(oof)
oof_rmse_log = mean_squared_error(y[mask], oof[mask], squared=False)
oof_rmse_units = mean_squared_error(np.expm1(y[mask]), np.expm1(oof[mask]), squared=False)
print("\n==== OOF performance (TF-IDF upgrade, no-names) ====")
print(f"OOF RMSE(log):   {oof_rmse_log:.4f}")
print(f"OOF RMSE(units): {oof_rmse_units:,.1f}")

# Save OOF
id_col = 'asin_key' if 'asin_key' in X.columns else ('product_page_url' if 'product_page_url' in X.columns else None)
oof_df = pd.DataFrame({
    'row_id': X.index,
    'id': X[id_col] if id_col else np.nan,
    'y_true': y,
    'y_pred': oof,
    'fold': X['fold']
})
oof_df.to_csv('oof_preds_tfidf_upgrade_no_names.csv', index=False)
print("✅ Saved: oof_preds_tfidf_upgrade_no_names.csv")

# Aggregate feature importances (indices only)
if len(imp_all):
    agg = pd.concat(imp_all, ignore_index=True).groupby('feature_idx', as_index=False)['gain'].mean()
    top50 = agg.sort_values('gain', ascending=False).head(50)
    top50.to_csv('feature_importance_top50_nonames.csv', index=False)
    print("\nTop 50 feature indices by mean gain (names omitted):")
    display(top50.head(20))
    print("✅ Saved: feature_importance_top50_nonames.csv")
else:
    print("\n[Note] No feature importances collected.")

print("\n🎯 TF-IDF upgrade training (no feature names) complete.")

# CatBoost text+tabular with time-based CV

In [ ]:
import os, gc, numpy as np, pandas as pd
from IPython.display import display

# --- install/import catboost ---
try:
    import catboost as cb
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost==1.2.5"])
    import catboost as cb

SEED = 42
np.random.seed(SEED)

# -----------------------------
# Load dfm & folds
# -----------------------------
if 'dfm' not in globals():
    dfm = pd.read_parquet('amazon_modeling_frame_cv.parquet')

assert {'y','product_title'}.issubset(dfm.columns), "dfm must include 'y' and 'product_title'."

# derive log1p_total_reviews if missing (used as numeric)
if 'log1p_total_reviews' not in dfm.columns and 'total_reviews' in dfm.columns:
    dfm['log1p_total_reviews'] = np.log1p(dfm['total_reviews'].fillna(0))

# use saved indices if present, else reconstruct from fold/time_bin
fold_ids = []
if os.path.exists('cv_splits_time_expanding.npz'):
    npz = np.load('cv_splits_time_expanding.npz', allow_pickle=True)
    i = 0
    while f'fold{i}_train_idx' in npz and f'fold{i}_valid_idx' in npz:
        tr = npz[f'fold{i}_train_idx']; va = npz[f'fold{i}_valid_idx']
        if len(tr) and len(va): fold_ids.append((tr, va))
        i += 1
else:
    assert {'fold','time_bin'}.issubset(dfm.columns), "Need either saved folds or dfm['fold','time_bin']."
    bins = sorted(dfm.loc[dfm['fold'] >= 0, 'time_bin'].unique().tolist())
    for i, k in enumerate(bins):
        va = dfm.index[dfm['fold'] == i].to_numpy()
        tr = dfm.index[dfm['time_bin'] < k].to_numpy()
        if len(tr) and len(va): fold_ids.append((tr, va))

assert len(fold_ids) > 0, "No folds found."

# -----------------------------
# Feature schema
# -----------------------------
# booleans -> uint8 for clarity (CatBoost handles numeric+cats natively)
bool_cols = [c for c in [
    'is_sponsored_fixed','is_best_seller_fixed','has_coupon_fixed',
    'buy_box_availability','has_discount','final_price_imputed',
    'used_variant_as_orig','no_featured_offer_flag'
] if c in dfm.columns]
for c in bool_cols:
    dfm[c] = dfm[c].fillna(False).astype('uint8')

num_cols = [c for c in [
    'product_rating',
    'log1p_total_reviews' if 'log1p_total_reviews' in dfm.columns else None,
    'log1p_final_price' if 'log1p_final_price' in dfm.columns else None,
    'final_price' if 'final_price' in dfm.columns else None,
    'discount_pct_capped' if 'discount_pct_capped' in dfm.columns else None,
] if c is not None]

cat_cols = [c for c in ['price_source','coarse_cat','brand_key','discount_bucket'] if c in dfm.columns]

text_col = 'product_title'

# ensure dtypes CatBoost-friendly
for c in cat_cols:
    dfm[c] = dfm[c].astype(str).fillna("NA")
dfm[text_col] = dfm[text_col].astype(str).fillna("")

# final feature order: numeric + booleans + categoricals + text
features = num_cols + bool_cols + cat_cols + [text_col]

# indices for CatBoost Pools
cat_feature_indices = [features.index(c) for c in cat_cols]
text_feature_indices = [features.index(text_col)]

# -----------------------------
# CatBoost params
# -----------------------------
params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 5.0,
    'random_seed': SEED,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'rsm': 0.8,  # column sampling for oblivious trees
    'early_stopping_rounds': 300,
    'iterations': 5000,
    'verbose': 200,
    'allow_writing_files': False,
}

# -----------------------------
# CV loop
# -----------------------------
y = dfm['y'].astype('float64').values
oof = np.full(len(dfm), np.nan, dtype=np.float64)
feat_imps = None
fold_summ = []

for i, (tr_idx, va_idx) in enumerate(fold_ids):
    print(f"\n==== CatBoost Fold {i} ({len(tr_idx)} train, {len(va_idx)} valid) ====")
    X_tr = dfm.iloc[tr_idx][features]
    X_va = dfm.iloc[va_idx][features]
    y_tr = y[tr_idx]; y_va = y[va_idx]

    train_pool = cb.Pool(
        X_tr, label=y_tr,
        cat_features=cat_feature_indices,
        text_features=text_feature_indices,
        feature_names=features
    )
    valid_pool = cb.Pool(
        X_va, label=y_va,
        cat_features=cat_feature_indices,
        text_features=text_feature_indices,
        feature_names=features
    )

    model = cb.CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    pred_va = model.predict(valid_pool)
    oof[va_idx] = pred_va

    # metrics (also show units; clip for numeric stability)
    rmse_log = np.sqrt(((pred_va - y_va) ** 2).mean())
    rmse_units = np.sqrt(((np.expm1(np.clip(pred_va, -1, 12)) - np.expm1(np.clip(y_va, -1, 12))) ** 2).mean())
    print(f"Fold {i} — RMSE(log): {rmse_log:.4f} | RMSE(units): {rmse_units:,.1f}")

    # feature importance
    imp_vals = model.get_feature_importance(train_pool, type='FeatureImportance')
    imp_df = pd.DataFrame({'feature': features, 'gain': imp_vals, 'fold': i})
    feat_imps = imp_df if feat_imps is None else pd.concat([feat_imps, imp_df], axis=0, ignore_index=True)

    # save model
    model.save_model(f'catboost_model_fold{i}.cbm')

    del X_tr, X_va, train_pool, valid_pool, model
    gc.collect()

# -----------------------------
# OOF summary
# -----------------------------
mask = ~np.isnan(oof)
oof_rmse_log = np.sqrt(((oof[mask] - y[mask]) ** 2).mean())
oof_rmse_units = np.sqrt(((np.expm1(np.clip(oof[mask], -1, 12)) - np.expm1(np.clip(y[mask], -1, 12))) ** 2).mean())

print("\n==== OOF performance — CatBoost ====")
print(f"OOF RMSE(log):   {oof_rmse_log:.4f}")
print(f"OOF RMSE(units): {oof_rmse_units:,.1f}")

# save OOF
id_col = 'asin_key' if 'asin_key' in dfm.columns else ('product_page_url' if 'product_page_url' in dfm.columns else None)
oof_df = pd.DataFrame({
    'row_id': dfm.index,
    'id': dfm[id_col] if id_col else np.nan,
    'y_true': y,
    'y_pred': oof,
    'fold': dfm.get('fold', -1)
})
oof_df.to_csv('oof_preds_catboost.csv', index=False)
print("✅ Saved: oof_preds_catboost.csv")

# aggregate & show top features
if feat_imps is not None and len(feat_imps):
    agg = feat_imps.groupby('feature', as_index=False)['gain'].mean().sort_values('gain', ascending=False)
    top = agg.head(30)
    top.to_csv('catboost_feature_importance.csv', index=False)
    print("\nTop features (mean gain across folds):")
    display(top)
    print("✅ Saved: catboost_feature_importance.csv")
else:
    print("\n[Note] No feature importances collected.")

print("\n🎯 CatBoost training complete.")

# FINAL MODEL — CatBoost (text + tabular) trained on FULL data

In [ ]:
import os, json, gc, numpy as np, pandas as pd
from datetime import datetime
from IPython.display import display

# --- import catboost (installed earlier) ---
import catboost as cb

# -----------------------------
# 0) Load dfm and ensure target
# -----------------------------
if 'dfm' not in globals():
    dfm = pd.read_parquet('amazon_modeling_frame_cv.parquet')

assert 'y' in dfm.columns and 'product_title' in dfm.columns, "dfm must contain ['y','product_title']"

# derive log1p_total_reviews if missing
if 'log1p_total_reviews' not in dfm.columns and 'total_reviews' in dfm.columns:
    dfm['log1p_total_reviews'] = np.log1p(dfm['total_reviews'].fillna(0))

# -----------------------------
# 1) Feature schema (same as best CV run)
# -----------------------------
bool_cols = [c for c in [
    'is_sponsored_fixed','is_best_seller_fixed','has_coupon_fixed',
    'buy_box_availability','has_discount','final_price_imputed',
    'used_variant_as_orig','no_featured_offer_flag'
] if c in dfm.columns]
for c in bool_cols:
    dfm[c] = dfm[c].fillna(False).astype('uint8')

num_cols = [c for c in [
    'product_rating',
    'log1p_total_reviews' if 'log1p_total_reviews' in dfm.columns else None,
    'log1p_final_price' if 'log1p_final_price' in dfm.columns else None,
    'final_price' if 'final_price' in dfm.columns else None,
    'discount_pct_capped' if 'discount_pct_capped' in dfm.columns else None,
] if c is not None]

cat_cols = [c for c in ['price_source','coarse_cat','brand_key','discount_bucket'] if c in dfm.columns]
text_col = 'product_title'

# CatBoost-friendly dtypes
for c in cat_cols:
    dfm[c] = dfm[c].astype(str).fillna("NA")
dfm[text_col] = dfm[text_col].astype(str).fillna("")

# final feature order: numeric + booleans + categoricals + text
features = num_cols + bool_cols + cat_cols + [text_col]
cat_feature_indices  = [features.index(c) for c in cat_cols]
text_feature_indices = [features.index(text_col)]

# rows to train
mask = dfm['y'].notna()
X_full = dfm.loc[mask, features]
y_full = dfm.loc[mask, 'y'].astype('float64').values

print("Training rows:", len(X_full))
print("Feature counts — num:", len(num_cols), "bool:", len(bool_cols), "cat:", len(cat_cols), "text: 1")
print("First 5 features:", features[:5], "... last:", features[-3:])

# -----------------------------
# 2) Determine reasonable iterations from fold models (if available)
# -----------------------------
best_iters = []
for i in range(10):
    path = f'catboost_model_fold{i}.cbm'
    if os.path.exists(path):
        m = cb.CatBoostRegressor()
        m.load_model(path)
        try:
            best_iters.append(int(m.tree_count_))
        except Exception:
            try:
                best_iters.append(int(m.get_tree_count()))
            except Exception:
                pass

if best_iters:
    base_iters = int(np.median(best_iters))
    final_iters = max(400, int(round(base_iters * 1.15)))  # a small bump for full-data fit
else:
    base_iters = None
    final_iters = 3000  # fallback

print(f"Fold models found: {len(best_iters)} | median iters={base_iters} -> using final iters={final_iters}")

# -----------------------------
# 3) Train final model on ALL data
# -----------------------------
params = {
    'loss_function': 'RMSE',
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 5.0,
    'random_seed': 42,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'rsm': 0.8,
    'iterations': final_iters,
    'verbose': 200,
    'allow_writing_files': False,
}

train_pool = cb.Pool(
    X_full, label=y_full,
    cat_features=cat_feature_indices,
    text_features=text_feature_indices,
    feature_names=features
)

final_model = cb.CatBoostRegressor(**params)
final_model.fit(train_pool, use_best_model=False)  # no eval set → train on full data

# Optional: show train RMSE (not a generalization metric)
pred_train = final_model.predict(train_pool)
rmse_train_log = float(np.sqrt(((pred_train - y_full) ** 2).mean()))
rmse_train_units = float(np.sqrt(((np.expm1(np.clip(pred_train, -1, 12)) - np.expm1(np.clip(y_full, -1, 12))) ** 2).mean()))
print(f"\nTrain RMSE(log): {rmse_train_log:.4f} | Train RMSE(units): {rmse_train_units:,.1f}")

# -----------------------------
# 4) Save artifacts
# -----------------------------
final_model.save_model('catboost_full.cbm')
with open('features_used.txt', 'w') as f:
    for feat in features:
        f.write(f"{feat}\n")

meta = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "rows": int(len(X_full)),
    "features": features,
    "num_cols": num_cols,
    "bool_cols": bool_cols,
    "cat_cols": cat_cols,
    "text_col": text_col,
    "params": params,
    "base_cv_tree_counts": best_iters,
    "final_tree_count": int(final_model.tree_count_) if hasattr(final_model, "tree_count_") else None,
}
with open('catboost_full_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("\n✅ Saved:")
print(" - catboost_full.cbm")
print(" - features_used.txt")
print(" - catboost_full_meta.json")

# -----------------------------
# 5) Inference helper
# -----------------------------
def predict_on_df(df_new: pd.DataFrame, model_path: str = 'catboost_full.cbm') -> pd.Series:
    """
    Returns:
        y_pred (log1p units) and prints an RMSE estimate if 'y' present; also returns units via .attrs['units_pred']
    """
    # copy to avoid mutation
    d = df_new.copy()

    # reconstruct minimal derived columns if missing
    if 'log1p_total_reviews' not in d.columns and 'total_reviews' in d.columns:
        d['log1p_total_reviews'] = np.log1p(pd.to_numeric(d['total_reviews'], errors='coerce').fillna(0))
    if 'log1p_final_price' not in d.columns and 'final_price' in d.columns:
        d['log1p_final_price'] = np.log1p(pd.to_numeric(d['final_price'], errors='coerce'))

    # ensure presence of all expected feature columns
    for c in bool_cols:
        if c not in d.columns:
            d[c] = 0
        d[c] = d[c].fillna(False).astype('uint8')

    for c in num_cols:
        if c not in d.columns:
            d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors='coerce')

    for c in cat_cols:
        if c not in d.columns:
            d[c] = "NA"
        d[c] = d[c].astype(str).fillna("NA")

    if text_col not in d.columns:
        d[text_col] = ""

    X_inf = d[features]
    pool = cb.Pool(
        X_inf,
        cat_features=[features.index(c) for c in cat_cols],
        text_features=[features.index(text_col)],
        feature_names=features
    )

    model = cb.CatBoostRegressor()
    model.load_model(model_path)
    y_pred = model.predict(pool)
    # also attach units prediction
    units = np.expm1(np.clip(y_pred, -1, 12))
    s = pd.Series(y_pred, index=d.index, name='y_pred')
    s.attrs['units_pred'] = units
    return s

print("\n🔮 Usage:")
print("pred = predict_on_df(dfm)        # returns log1p-units; units via pred.attrs['units_pred']")
print("pred.head()")

# PREDICT & SAVE — using catboost_full.cbm

In [ ]:
import os, re, json, unicodedata, numpy as np, pandas as pd
from IPython.display import display
import catboost as cb

# ---------- OPTIONAL: set a CSV path if you want to predict on a fresh file ----------
PATH_IF_NEEDED = None  # e.g., "/kaggle/input/.../amazon_products_sales_data_uncleaned.csv"

# ---------- 0) Load model + meta ----------
assert os.path.exists("catboost_full.cbm"), "catboost_full.cbm not found (run the FINAL MODEL cell first)."
with open("catboost_full_meta.json", "r") as f:
    META = json.load(f)

FEATURES = META["features"]
NUM_COLS  = META["num_cols"]
BOOL_COLS = META["bool_cols"]
CAT_COLS  = META["cat_cols"]
TEXT_COL  = META["text_col"]

print("Model loaded. Expecting features:", FEATURES)

# ---------- 1) Choose input frame ----------
if "dfd" in globals():
    df_in = dfd.copy()
    src = "dfd (deduped)"
elif "df" in globals():
    df_in = df.copy()
    src = "df (raw)"
elif PATH_IF_NEEDED:
    df_in = pd.read_csv(PATH_IF_NEEDED, low_memory=False)
    src = f"CSV: {PATH_IF_NEEDED}"
else:
    raise FileNotFoundError("No input found. Provide dfd/df in memory or set PATH_IF_NEEDED to a CSV.")

print(f"Using input: {src} | rows={len(df_in)}")

# ---------- 2) Helper funcs (brand, category, prices, ASIN, etc.) ----------
def extract_asin(u: object):
    if pd.isna(u): return None
    s = str(u).strip()
    m = re.search(r"/(?:dp|gp/product|product)/([A-Z0-9]{10})(?:[/?]|$)", s, re.I)
    if m: return m.group(1).upper()
    m = re.search(r"(?:^|[^A-Z0-9])([A-Z0-9]{10})(?:$|[^A-Z0-9])", s)
    if m: 
        cand = m.group(1).upper()
        if re.fullmatch(r"[A-Z0-9]{10}", cand): return cand
    return None

def infer_coarse_from_title(s: object) -> str:
    if pd.isna(s): return "Other"
    t = unicodedata.normalize("NFKC", str(s)).lower()
    rules = [
        (r'\b(laptop|notebook|macbook)\b', 'Laptop'),
        (r'\b(headphone|earbud|earphone|tws|headset|speaker|soundbar)\b', 'Audio'),
        (r'\b(camera|dslr|mirrorless|webcam|action cam|gopro)\b', 'Camera'),
        (r'\b(phone|iphone|android|smartphone|oneplus|samsung galaxy)\b', 'Mobile'),
        (r'\b(ssd|hard\s*drive|hdd|micro\s*sd|sd\s*card|flash\s*drive|pendrive|pen\s*drive)\b', 'Storage'),
        (r'\b(mouse|keyboard|monitor|router|adapter|charger|cable|hub|dock|power\s*bank)\b', 'Accessory'),
        (r'\b(tv|television|projector)\b', 'TV/Display'),
    ]
    for pat, lab in rules:
        if re.search(pat, t): return lab
    return "Other"

STOPWORDS = {'the','a','an','new','latest','2025','portable','wireless','with','for','and','by','from','brand'}
def extract_brand(title: object) -> str:
    if pd.isna(title): return 'unknown'
    s = unicodedata.normalize("NFKC", str(title)).strip()
    seg = re.split(r'[\-|–|:|\(|\[|,|/]', s, maxsplit=1)[0]
    m = re.search(r'^\s*(?:brand|manufacturer)\s*[:\-]\s*([A-Za-z0-9\-\+\. ]{2,})', seg, flags=re.I)
    if m: seg = m.group(1)
    tok = re.findall(r'[A-Za-z0-9\+\.\-]+', seg)
    if not tok: return 'unknown'
    cand = tok[0].lower()
    if cand in STOPWORDS or len(cand) <= 1:
        cand = tok[1].lower() if len(tok) > 1 else 'unknown'
    return cand

def parse_price(x):
    if pd.isna(x): return np.nan
    s = str(x)
    nums = re.findall(r'\d[\d,]*(?:\.\d+)?', s)
    if not nums: return np.nan
    try: return float(nums[0].replace(',', ''))
    except: return np.nan

# ---------- 3) Build the minimal features the model expects ----------
d = df_in.copy()

# IDs
if 'asin_key' not in d.columns:
    if 'product_page_url' in d.columns:
        d['asin_key'] = d['product_page_url'].map(extract_asin)
    else:
        d['asin_key'] = np.nan

# Ensure text + basic cols
if 'product_title' not in d.columns and 'title' in d.columns:
    d['product_title'] = d['title']
d['product_title'] = d.get('product_title', pd.Series([""]*len(d)))

# If price cols still raw strings, coerce
for c_old, c_new in [('current/discounted_price','discounted_price'),
                     ('listed_price','original_price'),
                     ('price_on_variant','price_on_variant')]:
    if c_old in d.columns and c_new not in d.columns:
        d[c_new] = d[c_old]

for c in ['discounted_price','original_price','price_on_variant']:
    if c in d.columns and not np.issubdtype(d[c].dtype, np.number):
        d[c] = d[c].map(parse_price).astype('float')

# coarse category & brand
if 'coarse_cat' not in d.columns:
    d['coarse_cat'] = d['product_title'].map(infer_coarse_from_title)
if 'brand_key' not in d.columns:
    d['brand_key'] = d['product_title'].map(extract_brand)

# price_source + final_price (discounted > variant > original)
cand_disc = d.get('discounted_price')
cand_var  = d.get('price_on_variant')
cand_orig = d.get('original_price')

if cand_disc is None: cand_disc = pd.Series([np.nan]*len(d))
if cand_var  is None: cand_var  = pd.Series([np.nan]*len(d))
if cand_orig is None: cand_orig = pd.Series([np.nan]*len(d))

d['final_price'] = cand_disc.where(cand_disc>0).combine_first(
                    cand_var.where(cand_var>0)).combine_first(
                    cand_orig.where(cand_orig>0))
d['price_source'] = np.select(
    [cand_disc.notna(), cand_disc.isna() & cand_var.notna(), cand_disc.isna() & cand_var.isna() & cand_orig.notna()],
    ['discounted','variant','original'], default='none'
)

# fill original_price when safely inferable from variant
thresh = 1.05
orig_filled = cand_orig.copy()
mask_fill = (orig_filled.isna() & cand_disc.notna() & cand_var.notna() & (cand_var > cand_disc*thresh))
orig_filled[mask_fill] = cand_var[mask_fill]
d['original_price_filled'] = orig_filled
d['used_variant_as_orig'] = mask_fill.fillna(False).astype('uint8')

# Discount % & buckets
disc_mask = d['original_price_filled'].notna() & d['final_price'].notna()
disc_raw = (d['original_price_filled'] - d['final_price']) / d['original_price_filled'] * 100
disc_fixed = disc_raw.where(disc_mask & (disc_raw >= 0) & (disc_raw <= 95))
d['discount_pct_capped'] = disc_fixed.clip(lower=0, upper=95)
bins = [-0.1,0,5,10,20,30,50,95,1e6]
labels = ['0','0-5','5-10','10-20','20-30','30-50','50-95','>95?']
try:
    d['discount_bucket'] = pd.cut(d['discount_pct_capped'], bins=bins, labels=labels, include_lowest=True)
except Exception:
    d['discount_bucket'] = pd.Categorical(['0']*len(d), categories=labels)

# Has discount boolean + log price
d['has_discount'] = (d['final_price'].notna() & d['original_price_filled'].notna() & (d['final_price'] < d['original_price_filled'])).astype('uint8')
d['log1p_final_price'] = np.log1p(d['final_price'])

# Reviews → log1p_total_reviews (if available)
if 'log1p_total_reviews' not in d.columns and 'total_reviews' in d.columns:
    d['log1p_total_reviews'] = np.log1p(pd.to_numeric(d['total_reviews'], errors='coerce').fillna(0))

# Booleans expected by the model (default to 0/False if missing)
for c in BOOL_COLS:
    if c not in d.columns:
        d[c] = 0
    d[c] = d[c].fillna(0).astype('uint8')

# Categorical columns as string
for c in CAT_COLS:
    if c not in d.columns:
        d[c] = "NA"
    d[c] = d[c].astype(str).fillna("NA")

# Text col
if TEXT_COL not in d.columns:
    d[TEXT_COL] = ""

# Ensure all expected features exist
missing_feats = [c for c in FEATURES if c not in d.columns]
if missing_feats:
    print("Creating missing numeric features as NaN:", missing_feats)
    for c in missing_feats:
        d[c] = np.nan

# Reorder columns for the model
X_inf = d[FEATURES].copy()

# ---------- 4) Build CatBoost Pool & Predict ----------
cat_feature_indices  = [FEATURES.index(c) for c in CAT_COLS if c in FEATURES]
text_feature_indices = [FEATURES.index(TEXT_COL)] if TEXT_COL in FEATURES else []

pool = cb.Pool(
    X_inf,
    cat_features=cat_feature_indices,
    text_features=text_feature_indices,
    feature_names=FEATURES
)

model = cb.CatBoostRegressor()
model.load_model("catboost_full.cbm")
y_log = model.predict(pool)

# Back-transform to units (stable)
pred_units = np.expm1(np.clip(y_log, -1, 12))
pred_units_rounded = np.rint(pred_units).astype(int)

# ---------- 5) Assemble & save submission ----------
id_col = "asin_key" if "asin_key" in d.columns else ("product_page_url" if "product_page_url" in d.columns else None)
if id_col is None:
    d['row_id'] = np.arange(len(d))
    id_col = 'row_id'

sub = pd.DataFrame({
    'id': d[id_col],
    'predicted_units': pred_units,
    'predicted_units_rounded': pred_units_rounded,
    'y_pred_log': y_log
})

sub.to_csv("amazon_predictions.csv", index=False)
print("\n✅ Saved -> amazon_predictions.csv")
print(sub.head())

<div style="background-color:#166534; color:#fde047; padding:20px; border-radius:10px;">
  <h2 style="font-size:22px; font-family:calibri;"><b>✅ Summary & Takeaways</b></h2>
  <ul style="font-size:18px; font-family:calibri; line-height:1.8em;">
    <li>✅ <b>Final Model:</b> CatBoost with native text + tabular → RMSE ≈ 0.996</li>
    <li>🔍 <b>Top Signals:</b> log1p(reviews), final_price, title text</li>
    <li>🛠 <b>Data Cleaning:</b> was more impactful than feature count</li>
    <li>🧪 <b>ASIN-safe deduplication:</b> crucial for product identity</li>
    <li>📈 <b>Title-based semantic cues</b> worked better in CatBoost than TF-IDF</li>
  </ul>
</div>

<div style="background-color:#166534; color:#fde047; padding: 20px; border-radius: 10px;">
  <h2 style="font-size:22px; font-family:calibri;"><b>🙌 Support & Share</b></h2>
  <ul style="font-size:18px; font-family:calibri; line-height:1.8em;">
    <li>⭐ If this helped, upvote & follow the notebook</li>
    <li>🔁 Fork it — try stacking, category models, or new text embeddings</li>
    <li>💬 Comment your ideas — I read and reply to every one!</li>
  </ul>
  <p style="font-size:18px; font-family:calibri;">
    Let’s build smarter e-commerce systems — one cleaned ASIN at a time. 🛒📈🔍
  </p>
</div>
